In [98]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

import torch
import yaml

from engine.model_adapter import ModelAdapter
from engine.kv_cache import KVCache
from engine.hf_kv_cache import EngineKVCache


with open("configs/inference.yaml", "r") as f:
    config = yaml.safe_load(f)




import gc
import torch

# Delete model-related Python objects
for name in [
    "adapter",
    "model",
    "attention",
    "hf_cache",
    "kv_cache",
    "full_output",
    "uncached_k",
    "uncached_v",
]:
    if name in globals():
        del globals()[name]

# Run Python garbage collection
gc.collect()

# Release PyTorch's cached CUDA memory
torch.cuda.empty_cache()

# Ask CUDA to release unused IPC memory
torch.cuda.ipc_collect()

print(torch.cuda.memory_summary())


adapter = ModelAdapter(
    model_name=config["model"]["model_name"],
    device=config["device"],
)

model = adapter.model
model.eval()

print("device:", adapter.device)
print("dtype:", model.dtype)
print("layers:", model.config.num_hidden_layers)
print("KV heads:", model.config.num_key_value_heads)

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |    968 MiB |   2909 MiB |   8861 MiB |   7892 MiB |
|       from large pool |    956 MiB |   2837 MiB |   3798 MiB |   2842 MiB |
|       from small pool |     12 MiB |     73 MiB |   5062 MiB |   5050 MiB |
|---------------------------------------------------------------------------|
| Active memory         |    968 MiB |   2909 MiB |   8861 MiB |   7892 MiB |
|       from large pool |    956 MiB |   2837 MiB |   3798 MiB |

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

device: cuda
dtype: torch.bfloat16
layers: 24
KV heads: 2


In [99]:
prompt = "The capital of France is"

prompt_ids = adapter.tokenize(prompt)

input_ids = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=adapter.device,
)


T = input_ids.shape[1]

print("prompt length:", T)
print("prompt ids:", prompt_ids)
print("input ids:", input_ids)

prompt length: 5
prompt ids: [785, 6722, 315, 9625, 374]
input ids: tensor([[ 785, 6722,  315, 9625,  374]], device='cuda:0')


In [100]:
with torch.no_grad():
    prompt_output = model(
        input_ids=input_ids,
        use_cache=False,
    )

next_token = prompt_output.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)
print("prompt_output.logits:", prompt_output.logits)
print("prompt_output.logits.shape", prompt_output.logits.shape)

print("next token:", next_token.item())

prompt_output.logits: tensor([[[ 2.0938, -0.0251, -1.0781,  ..., -1.9922, -1.9922, -1.9922],
         [ 5.6875,  4.9062,  1.0625,  ..., -1.8906, -1.8906, -1.8906],
         [ 2.5781, -0.5078, -2.7188,  ..., -5.0000, -5.0000, -5.0000],
         [10.0000,  6.7188,  3.7344,  ..., -3.7969, -3.7969, -3.7969],
         [ 7.5938,  4.1250,  3.9219,  ..., -4.6562, -4.6562, -4.6562]]],
       device='cuda:0', dtype=torch.bfloat16)
prompt_output.logits.shape torch.Size([1, 5, 151936])
next token: 12095


Run the uncached path and capture every layer

In [28]:
full_input = torch.cat(
    [input_ids, next_token],
    dim=1,
)

uncached_layers = {}

hooks = []

for i, layer in enumerate(model.model.layers):

    def make_hook(layer_idx):
        def hook(module, inputs, output):

            hidden = (
                output[0]
                if isinstance(output, tuple)
                else output
            )

            # Only keep the NEW token at position T
            uncached_layers[layer_idx] = (
                hidden[:, -1, :]
                .detach()
                .float()
                .cpu()
            )

        return hook

    hooks.append(
        layer.register_forward_hook(
            make_hook(i)
        )
    )

with torch.no_grad():
    uncached_output = model(
        input_ids=full_input,
        use_cache=False,
    )

for hook in hooks:
    hook.remove()



print('uncached_output.logtis:', uncached_output.logits)
print('uncached_output.logits.shape:', uncached_output.logits.shape)
print("Captured", len(uncached_layers), "uncached layers")


uncached_output.logtis: tensor([[[ 2.0938, -0.0251, -1.0781,  ..., -1.9922, -1.9922, -1.9922],
         [ 5.6875,  4.9062,  1.0625,  ..., -1.8906, -1.8906, -1.8906],
         [ 2.5781, -0.5078, -2.7188,  ..., -5.0000, -5.0000, -5.0000],
         [10.0000,  6.7188,  3.7344,  ..., -3.7969, -3.7969, -3.7969],
         [ 7.5938,  4.1250,  3.9219,  ..., -4.6562, -4.6562, -4.6562],
         [12.1250,  7.4062,  5.2188,  ..., -2.6719, -2.6719, -2.6719]]],
       device='cuda:0', dtype=torch.bfloat16)
uncached_output.logits.shape: torch.Size([1, 6, 151936])
Captured 24 uncached layers


In [29]:
uncached_output.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True
)

tensor([[13]], device='cuda:0')

the engine cache and prefill

In [101]:
kv_cache = KVCache(
    num_layers=model.config.num_hidden_layers,
    num_kv_heads=model.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model.config.hidden_size
        // model.config.num_attention_heads
    ),
    dtype=model.dtype,
    device=adapter.device,
)

hf_cache = EngineKVCache(kv_cache)

print("cache length before prefill:", hf_cache.get_seq_length())
with torch.no_grad():
    prefill_output = model(
        input_ids=input_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )

print("cache length after prefill:", hf_cache.get_seq_length())
print("prefill_output.logits:", prefill_output.logits)
print("prefill_output.logits.shape:", prefill_output.logits.shape)
print("cache length after prefill:", hf_cache.get_seq_length())

cache length before prefill: 0
cache length after prefill: 5
prefill_output.logits: tensor([[[ 2.0938, -0.0251, -1.0781,  ..., -1.9922, -1.9922, -1.9922],
         [ 5.6875,  4.9062,  1.0625,  ..., -1.8906, -1.8906, -1.8906],
         [ 2.5781, -0.5078, -2.7188,  ..., -5.0000, -5.0000, -5.0000],
         [10.0000,  6.7188,  3.7344,  ..., -3.7969, -3.7969, -3.7969],
         [ 7.5938,  4.1250,  3.9219,  ..., -4.6562, -4.6562, -4.6562]]],
       device='cuda:0', dtype=torch.bfloat16)
prefill_output.logits.shape: torch.Size([1, 5, 151936])
cache length after prefill: 5


In [31]:
prefill_output.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

tensor([[12095]], device='cuda:0')

Capture every layer during cached decode

In [32]:
cached_layers = {}

hooks = []

for i, layer in enumerate(model.model.layers):

    def make_hook(layer_idx):
        def hook(module, inputs, output):

            hidden = (
                output[0]
                if isinstance(output, tuple)
                else output
            )

            cached_layers[layer_idx] = (
                hidden[:, -1, :]
                .detach()
                .float()
                .cpu()
            )

        return hook

    hooks.append(
        layer.register_forward_hook(
            make_hook(i)
        )
    )

position_ids = torch.tensor(
    [[T]],
    dtype=torch.long,
    device=adapter.device,
)

with torch.no_grad():
    cached_output = model(
        input_ids=next_token,
        position_ids=position_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )

for hook in hooks:
    hook.remove()



print("cached_output.logits:", cached_output.logits)
print("cached_output.logits.shape:", cached_output.logits.shape)
print("Captured", len(cached_layers), "cached layers")
print("cache length after decode:", hf_cache.get_seq_length())

cached_output.logits: tensor([[[12.1250,  7.3750,  5.1562,  ..., -2.6250, -2.6250, -2.6250]]],
       device='cuda:0', dtype=torch.bfloat16)
cached_output.logits.shape: torch.Size([1, 1, 151936])
Captured 24 cached layers
cache length after decode: 6


In [33]:
cached_output.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

tensor([[13]], device='cuda:0')

In [34]:
print("=" * 80)
print("CACHED vs UNCACHED")
print("=" * 80)

layer_diffs = []

for i in range(model.config.num_hidden_layers):

    diff = (
        cached_layers[i]
        - uncached_layers[i]
    ).abs()

    max_diff = diff.max().item()
    mean_diff = diff.mean().item()

    layer_diffs.append(
        (i, max_diff, mean_diff)
    )

    print(
        f"layer {i:02d} | "
        f"max={max_diff:.8e} | "
        f"mean={mean_diff:.8e}"
    )

CACHED vs UNCACHED
layer 00 | max=3.90625000e-03 | mean=2.19626090e-04
layer 01 | max=7.81250000e-03 | mean=5.74248203e-04
layer 02 | max=7.81250000e-03 | mean=9.42570798e-04
layer 03 | max=7.81250000e-03 | mean=1.38749392e-03
layer 04 | max=3.12500000e-02 | mean=2.82730372e-03
layer 05 | max=1.56250000e-02 | mean=2.54394324e-03
layer 06 | max=1.56250000e-02 | mean=2.83476291e-03
layer 07 | max=1.56250000e-02 | mean=2.88554607e-03
layer 08 | max=3.12500000e-02 | mean=2.92178569e-03
layer 09 | max=2.34375000e-02 | mean=2.91912886e-03
layer 10 | max=2.34375000e-02 | mean=3.24365078e-03
layer 11 | max=3.12500000e-02 | mean=3.47566605e-03
layer 12 | max=3.12500000e-02 | mean=3.43336374e-03
layer 13 | max=3.12500000e-02 | mean=3.48602026e-03
layer 14 | max=6.25000000e-02 | mean=3.70549294e-03
layer 15 | max=6.25000000e-02 | mean=4.10570437e-03
layer 16 | max=1.25000000e-01 | mean=5.35641378e-03
layer 17 | max=1.25000000e-01 | mean=5.92276035e-03
layer 18 | max=4.10156250e-02 | mean=6.272111

In [35]:
uncached_logits = uncached_output.logits[:, -1, :]
cached_logits = cached_output.logits[:, -1, :]

logit_diff = (
    cached_logits.float()
    - uncached_logits.float()
).abs()

print()
print("LOGITS")
print("max :", logit_diff.max().item())
print("mean:", logit_diff.mean().item())

print()
print("cached token  :", cached_logits.argmax(dim=-1).item())
print("uncached token:", uncached_logits.argmax(dim=-1).item())


LOGITS
max : 0.173583984375
mean: 0.027369718998670578

cached token  : 13
uncached token: 13


In [36]:
layer = 0

K = hf_cache.kv_cache.key_cache[layer]
V = hf_cache.kv_cache.value_cache[layer]

print("K shape:", K.shape)
print("V shape:", V.shape)

print("valid K:", K[:, :, :T+1, :].shape)
print("valid V:", V[:, :, :T+1, :].shape)

K shape: torch.Size([1, 2, 6, 64])
V shape: torch.Size([1, 2, 6, 64])
valid K: torch.Size([1, 2, 6, 64])
valid V: torch.Size([1, 2, 6, 64])


#1: compare cached K/V against uncached K/V

In [37]:
import inspect
# model.model.layers[0].self_attn.forward
print(inspect.getsource(
    model.model.layers[0].self_attn.forward
))

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: torch.Tensor | None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[FlashAttentionKwargs],
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_values is not None:
            key_states, value_states = past_key_values.update(key_states, value_states, self.layer_idx)

        attention_interface: Callable = AL

In [38]:
# print(inspect.getsource(
#     model.model.layers[0].self_attn.k_proj
# ))

model.model.layers[0].self_attn.k_proj

Linear(in_features=896, out_features=128, bias=True)

In [39]:
from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb

num_kv_heads = model.config.num_key_value_heads
num_heads = model.config.num_attention_heads
head_dim = model.config.hidden_size // model.config.num_attention_heads



layer_idx = 0
attention = model.model.layers[layer_idx].self_attn

uncached_k = {}
uncached_v = {}


original_forward = attention.forward


def debug_forward(*args, **kwargs):

    # hidden_states is the first positional argument
    hidden_states = args[0] if args else kwargs["hidden_states"]

    B, T, _ = hidden_states.shape

    # ---------------------------------------------------------
    # Same projections used by Qwen2
    # ---------------------------------------------------------

    key_states = attention.k_proj(hidden_states)
    value_states = attention.v_proj(hidden_states)

    key_states = key_states.view(
        B,
        T,
        num_kv_heads,
        head_dim,
    ).transpose(1, 2)

    value_states = value_states.view(
        B,
        T,
        num_kv_heads,
        head_dim,
    ).transpose(1, 2)

    # ---------------------------------------------------------
    # Get position embeddings
    # ---------------------------------------------------------

    position_embeddings = kwargs.get(
        "position_embeddings",
        None,
    )

    if position_embeddings is None:
        raise RuntimeError(
            "position_embeddings not found"
        )

    cos, sin = position_embeddings

    # ---------------------------------------------------------
    # Apply RoPE exactly as Qwen2 does
    # ---------------------------------------------------------

    dummy_query = torch.zeros(
        B,
        num_heads,
        T,
        attention.head_dim,
        dtype=hidden_states.dtype,
        device=hidden_states.device,
    )

    dummy_query, key_states = apply_rotary_pos_emb(
        dummy_query,
        key_states,
        cos,
        sin,
    )

    # ---------------------------------------------------------
    # Save K/V
    # ---------------------------------------------------------

    uncached_k["value"] = key_states.detach().clone()
    uncached_v["value"] = value_states.detach().clone()

    # Run original attention
    return original_forward(*args, **kwargs)


attention.forward = debug_forward

In [40]:
full_input = torch.cat(
    [input_ids, next_token],
    dim=1,
)

with torch.no_grad():
    full_output = model(
        input_ids=full_input,
        use_cache=False,
    )

attention.forward = original_forward

In [41]:
print("uncached K:", uncached_k["value"].shape)
print("uncached V:", uncached_v["value"].shape)

uncached K: torch.Size([1, 2, 6, 64])
uncached V: torch.Size([1, 2, 6, 64])


In [42]:
cached_k = hf_cache.kv_cache.key_cache[0][
    :, :, :T + 1, :
]

cached_v = hf_cache.kv_cache.value_cache[0][
    :, :, :T + 1, :
]


print("cached K:", cached_k.shape)
print("cached V:", cached_v.shape)

cached K: torch.Size([1, 2, 6, 64])
cached V: torch.Size([1, 2, 6, 64])


In [43]:
k_diff = (
    cached_k.float()
    - uncached_k["value"].float()
).abs()

v_diff = (
    cached_v.float()
    - uncached_v["value"].float()
).abs()

print("=" * 70)
print("K/V NUMERICAL EQUIVALENCE — LAYER 0")
print("=" * 70)

print()
print("K:")
print("max :", k_diff.max().item())
print("mean:", k_diff.mean().item())

print()
print("V:")
print("max :", v_diff.max().item())
print("mean:", v_diff.mean().item())

K/V NUMERICAL EQUIVALENCE — LAYER 0

K:
max : 0.0
mean: 0.0

V:
max : 0.0
mean: 0.0


In [44]:
print()
print("POSITION-BY-POSITION")
print("=" * 70)

for pos in range(T + 1):

    k_pos_diff = (
        cached_k[:, :, pos, :].float()
        - uncached_k["value"][:, :, pos, :].float()
    ).abs()

    v_pos_diff = (
        cached_v[:, :, pos, :].float()
        - uncached_v["value"][:, :, pos, :].float()
    ).abs()

    print(
        f"position {pos}: "
        f"K max={k_pos_diff.max().item():.8e}, "
        f"K mean={k_pos_diff.mean().item():.8e}, "
        f"V max={v_pos_diff.max().item():.8e}, "
        f"V mean={v_pos_diff.mean().item():.8e}"
    )


POSITION-BY-POSITION
position 0: K max=0.00000000e+00, K mean=0.00000000e+00, V max=0.00000000e+00, V mean=0.00000000e+00
position 1: K max=0.00000000e+00, K mean=0.00000000e+00, V max=0.00000000e+00, V mean=0.00000000e+00
position 2: K max=0.00000000e+00, K mean=0.00000000e+00, V max=0.00000000e+00, V mean=0.00000000e+00
position 3: K max=0.00000000e+00, K mean=0.00000000e+00, V max=0.00000000e+00, V mean=0.00000000e+00
position 4: K max=0.00000000e+00, K mean=0.00000000e+00, V max=0.00000000e+00, V mean=0.00000000e+00
position 5: K max=0.00000000e+00, K mean=0.00000000e+00, V max=0.00000000e+00, V mean=0.00000000e+00


This proves:

- ✅ Fixed pre-allocated cache
- ✅ Correct [B, H_kv, T, D] layout
- ✅ Correct cache positions
- ✅ Correct K/V writes
- ✅ Correct RoPE position for K
- ✅ Correct K/V prefix after decode


compare the query for the decoded token directly.

$$
Q_{cached}[:, :, 0, :] =? Q_{uncached}[:, :, 5, :]
$$

In [63]:
# ------------------------------------------------------------
# Capture Q from layer 0 during UNCACHED full forward
# ------------------------------------------------------------

num_kv_heads = model.config.num_key_value_heads
num_heads = model.config.num_attention_heads
head_dim = model.config.hidden_size // model.config.num_attention_heads


layer_idx = 0
attention = model.model.layers[layer_idx].self_attn

uncached_q = {}

original_forward = attention.forward


def debug_q_forward(*args, **kwargs):

    hidden_states = (
        args[0]
        if args
        else kwargs["hidden_states"]
    )

    B, T_full, _ = hidden_states.shape

    # Q projection
    query_states = attention.q_proj(hidden_states)

    query_states = query_states.view(
        B,
        T_full,
        num_heads,
        head_dim,
    ).transpose(1, 2)

    # Qwen2 applies RoPE to Q as well.
    position_embeddings = kwargs.get(
        "position_embeddings",
        None,
    )

    if position_embeddings is None:
        raise RuntimeError(
            "position_embeddings not found"
        )

    cos, sin = position_embeddings

    # We need the same RoPE operation.
    # Use a dummy K with the correct KV-head shape.
    dummy_k = torch.zeros(
        B,
        num_kv_heads,
        T_full,
        head_dim,
        dtype=hidden_states.dtype,
        device=hidden_states.device,
    )

    query_states, _ = apply_rotary_pos_emb(
        query_states,
        dummy_k,
        cos,
        sin,
    )

    uncached_q["value"] = (
        query_states
        .detach()
        .clone()
    )

    return original_forward(*args, **kwargs)


attention.forward = debug_q_forward

In [64]:
full_input = torch.cat(
    [input_ids, next_token],
    dim=1,
)

with torch.no_grad():
    _ = model(
        input_ids=full_input,
        use_cache=False,
    )

attention.forward = original_forward

print(
    "uncached Q shape:",
    uncached_q["value"].shape,
)

uncached Q shape: torch.Size([1, 14, 6, 64])


In [65]:
cached_q = {}

original_forward = attention.forward


def debug_cached_q_forward(*args, **kwargs):

    hidden_states = (
        args[0]
        if args
        else kwargs["hidden_states"]
    )

    B, T_decode, _ = hidden_states.shape

    query_states = attention.q_proj(hidden_states)

    query_states = query_states.view(
        B,
        T_decode,
        num_heads,
        head_dim,
    ).transpose(1, 2)

    position_embeddings = kwargs.get(
        "position_embeddings",
        None,
    )

    if position_embeddings is None:
        raise RuntimeError(
            "position_embeddings not found"
        )

    cos, sin = position_embeddings

    dummy_k = torch.zeros(
        B,
        num_kv_heads,
        T_decode,
        head_dim,
        dtype=hidden_states.dtype,
        device=hidden_states.device,
    )

    query_states, _ = apply_rotary_pos_emb(
        query_states,
        dummy_k,
        cos,
        sin,
    )

    cached_q["value"] = (
        query_states
        .detach()
        .clone()
    )

    return original_forward(*args, **kwargs)


attention.forward = debug_cached_q_forward

In [66]:
decode_position = input_ids.shape[1]

position_ids = torch.tensor(
    [[decode_position]],
    dtype=torch.long,
    device=adapter.device,
)

with torch.no_grad():
    _ = model(
        input_ids=next_token,
        position_ids=position_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )

attention.forward = original_forward

print(
    "cached Q shape:",
    cached_q["value"].shape,
)

cached Q shape: torch.Size([1, 14, 1, 64])


In [67]:
# Uncached Q for the NEW token = position 5
q_uncached = uncached_q["value"][:, :, -1, :]

# Cached Q has only one query token
q_cached = cached_q["value"][:, :, 0, :]

q_diff = (
    q_cached.float()
    - q_uncached.float()
).abs()

print("=" * 70)
print("Q NUMERICAL EQUIVALENCE — LAYER 0")
print("=" * 70)

print("cached Q shape:  ", q_cached.shape)
print("uncached Q shape:", q_uncached.shape)

print()
print("MAX DIFF :", q_diff.max().item())
print("MEAN DIFF:", q_diff.mean().item())

Q NUMERICAL EQUIVALENCE — LAYER 0
cached Q shape:   torch.Size([1, 14, 64])
uncached Q shape: torch.Size([1, 14, 64])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [68]:
import torch




num_kv_heads = model.config.num_key_value_heads
num_heads = model.config.num_attention_heads
head_dim = model.config.hidden_size // model.config.num_attention_heads

layer_idx = 0
attention = model.model.layers[layer_idx].self_attn

uncached_scores = {}

original_forward = attention.forward


def capture_uncached_scores(*args, **kwargs):

    hidden_states = (
        args[0]
        if args
        else kwargs["hidden_states"]
    )

    B, T, _ = hidden_states.shape

    # ------------------------------------------------
    # Q
    # ------------------------------------------------

    q = attention.q_proj(hidden_states)

    q = q.view(
        B,
        T,
        num_heads,
        head_dim,
    ).transpose(1, 2)

    # ------------------------------------------------
    # K
    # ------------------------------------------------

    k = attention.k_proj(hidden_states)

    k = k.view(
        B,
        T,
        num_kv_heads,
        head_dim,
    ).transpose(1, 2)

    # ------------------------------------------------
    # RoPE
    # ------------------------------------------------

    cos, sin = kwargs["position_embeddings"]

    q, k = apply_rotary_pos_emb(
        q,
        k,
        cos,
        sin,
    )

    # ------------------------------------------------
    # GQA
    # ------------------------------------------------

    k = k.repeat_interleave(
        num_heads
        // num_kv_heads,
        dim=1,
    )

    # ------------------------------------------------
    # RAW ATTENTION SCORES
    # ------------------------------------------------

    scores = torch.matmul(
        q,
        k.transpose(-1, -2),
    ) * attention.scaling

    uncached_scores["value"] = (
        scores.detach().clone()
    )

    return original_forward(*args, **kwargs)


attention.forward = capture_uncached_scores

In [69]:
full_input = torch.cat(
    [input_ids, next_token],
    dim=1,
)

with torch.no_grad():
    _ = model(
        input_ids=full_input,
        use_cache=False,
    )

attention.forward = original_forward

print(
    "uncached scores:",
    uncached_scores["value"].shape,
)

uncached scores: torch.Size([1, 14, 6, 6])


In [70]:
# Cached Q for the decode token
q_cached = cached_q["value"]

# Cached K for positions 0..5
k_cached = hf_cache.kv_cache.key_cache[0][
    :, :, :6, :
]

# GQA
k_cached = k_cached.repeat_interleave(
    num_heads
    // num_kv_heads,
    dim=1,
)

cached_scores = torch.matmul(
    q_cached,
    k_cached.transpose(-1, -2),
) * attention.scaling

print(
    "cached scores:",
    cached_scores.shape,
)

cached scores: torch.Size([1, 14, 1, 6])


In [71]:
scores_uncached = uncached_scores["value"][
    :, :, -1:, :
]

scores_cached = cached_scores

score_diff = (
    scores_cached.float()
    - scores_uncached.float()
).abs()

print("=" * 70)
print("RAW ATTENTION SCORE EQUIVALENCE — LAYER 0")
print("=" * 70)

print()
print("cached:  ", scores_cached.shape)
print("uncached:", scores_uncached.shape)

print()
print("MAX DIFF :", score_diff.max().item())
print("MEAN DIFF:", score_diff.mean().item())

RAW ATTENTION SCORE EQUIVALENCE — LAYER 0

cached:   torch.Size([1, 14, 1, 6])
uncached: torch.Size([1, 14, 1, 6])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [72]:
print()
print("POSITION-BY-POSITION")
print("=" * 70)

for pos in range(6):

    diff = (
        scores_cached[:, :, 0, pos].float()
        - scores_uncached[:, :, 0, pos].float()
    ).abs()

    print(
        f"key position {pos}: "
        f"max={diff.max().item():.8e} "
        f"mean={diff.mean().item():.8e}"
    )


POSITION-BY-POSITION
key position 0: max=0.00000000e+00 mean=0.00000000e+00
key position 1: max=0.00000000e+00 mean=0.00000000e+00
key position 2: max=0.00000000e+00 mean=0.00000000e+00
key position 3: max=0.00000000e+00 mean=0.00000000e+00
key position 4: max=0.00000000e+00 mean=0.00000000e+00
key position 5: max=0.00000000e+00 mean=0.00000000e+00


uncached_K

In [78]:
# ============================================================
# CAPTURE EXACT UNCACHED Q/K FOR LAYER 0
# ============================================================

layer_idx = 0
attention = model.model.layers[layer_idx].self_attn

uncached_q = {}
uncached_k = {}

original_forward = attention.forward


def capture_qk(*args, **kwargs):

    hidden_states = (
        args[0]
        if args
        else kwargs["hidden_states"]
    )

    position_embeddings = kwargs["position_embeddings"]

    B, T, _ = hidden_states.shape

    # --------------------------------------------------------
    # Q projection
    # --------------------------------------------------------

    q = attention.q_proj(hidden_states)

    q = q.view(
        B,
        T,
        num_heads,
        head_dim,
    ).transpose(1, 2)

    # --------------------------------------------------------
    # K projection
    # --------------------------------------------------------

    k = attention.k_proj(hidden_states)

    k = k.view(
        B,
        T,
        num_kv_heads,
        head_dim,
    ).transpose(1, 2)

    # --------------------------------------------------------
    # RoPE
    # --------------------------------------------------------

    cos, sin = position_embeddings

    q, k = apply_rotary_pos_emb(
        q,
        k,
        cos,
        sin,
    )

    # --------------------------------------------------------
    # SAVE EXACT TENSORS
    # --------------------------------------------------------

    uncached_q["value"] = q.detach().clone()

    uncached_k["value"] = k.detach().clone()

    return original_forward(*args, **kwargs)


attention.forward = capture_qk

In [79]:
# ============================================================
# UNCACHED FULL SEQUENCE
# ============================================================


full_input_ids = torch.cat(
    [
        input_ids,
        next_token
    ],
    dim=1,
)

with torch.no_grad():

    full_outputs = model(
        input_ids=full_input_ids,
        use_cache=False,
    )

attention.forward = original_forward

In [80]:
print("UNCACHED Q:", uncached_q["value"].shape)
print("UNCACHED K:", uncached_k["value"].shape)

UNCACHED Q: torch.Size([1, 14, 6, 64])
UNCACHED K: torch.Size([1, 2, 6, 64])


test the actual tensors directly.

In [73]:
hf_cache.kv_cache.key_cache[0][:, :, :6, :].shape

torch.Size([1, 2, 6, 64])

In [81]:
# ============================================================
# DIRECT Q/K SELF-ATTENTION CHECK
# ============================================================

print("=" * 70)
print("DIRECT Q/K CHECK — DECODE POSITION 5")
print("=" * 70)

# ------------------------------------------------------------
# Cached tensors
# ------------------------------------------------------------

q_c = cached_q["value"][:, :, 0, :]       # [1, H, D]

k_c = hf_cache.kv_cache.key_cache[0][
    :, :, :6, :
]                                         # [1, H_kv, 6, D]

# GQA
k_c = k_c.repeat_interleave(
    num_heads
    // num_kv_heads,
    dim=1,
)

k_c = k_c[:, :, :, :]                      # [1, H, 6, D]

# ------------------------------------------------------------
# Uncached tensors
# ------------------------------------------------------------

q_u = uncached_q["value"][:, :, -1, :]    # [1, H, D]

k_u = uncached_k["value"][:, :, :, :]     # [1, H_kv, 6, D]

k_u = k_u.repeat_interleave(
    num_heads
    // num_kv_heads,
    dim=1,
)

# ------------------------------------------------------------
# Q equivalence
# ------------------------------------------------------------

q_diff = (
    q_c.float() - q_u.float()
).abs()

print()
print("Q")
print("max :", q_diff.max().item())
print("mean:", q_diff.mean().item())

# ------------------------------------------------------------
# K equivalence
# ------------------------------------------------------------

k_diff = (
    k_c.float() - k_u.float()
).abs()

print()
print("K")
print("max :", k_diff.max().item())
print("mean:", k_diff.mean().item())

# ------------------------------------------------------------
# Compare each K position
# ------------------------------------------------------------

print()
print("K POSITION DIFFERENCES")

for pos in range(6):

    diff = (
        k_c[:, :, pos, :].float()
        - k_u[:, :, pos, :].float()
    ).abs()

    print(
        f"position {pos}: "
        f"max={diff.max().item():.8e} "
        f"mean={diff.mean().item():.8e}"
    )

# ------------------------------------------------------------
# Direct QK score
# ------------------------------------------------------------

scores_c = torch.einsum(
    "bhd,bhtd->bht",
    q_c,
    k_c,
) * attention.scaling

scores_u = torch.einsum(
    "bhd,bhtd->bht",
    q_u,
    k_u,
) * attention.scaling

score_diff = (
    scores_c.float() - scores_u.float()
).abs()

print()
print("DIRECT QK SCORES")

print(
    "max :",
    score_diff.max().item(),
)

print(
    "mean:",
    score_diff.mean().item(),
)

print()
print("POSITION-BY-POSITION")

for pos in range(6):

    diff = (
        scores_c[:, :, pos].float()
        - scores_u[:, :, pos].float()
    ).abs()

    print(
        f"position {pos}: "
        f"max={diff.max().item():.8e} "
        f"mean={diff.mean().item():.8e}"
    )

DIRECT Q/K CHECK — DECODE POSITION 5

Q
max : 0.0
mean: 0.0

K
max : 0.0
mean: 0.0

K POSITION DIFFERENCES
position 0: max=0.00000000e+00 mean=0.00000000e+00
position 1: max=0.00000000e+00 mean=0.00000000e+00
position 2: max=0.00000000e+00 mean=0.00000000e+00
position 3: max=0.00000000e+00 mean=0.00000000e+00
position 4: max=0.00000000e+00 mean=0.00000000e+00
position 5: max=0.00000000e+00 mean=0.00000000e+00

DIRECT QK SCORES
max : 0.0
mean: 0.0

POSITION-BY-POSITION
position 0: max=0.00000000e+00 mean=0.00000000e+00
position 1: max=0.00000000e+00 mean=0.00000000e+00
position 2: max=0.00000000e+00 mean=0.00000000e+00
position 3: max=0.00000000e+00 mean=0.00000000e+00
position 4: max=0.00000000e+00 mean=0.00000000e+00
position 5: max=0.00000000e+00 mean=0.00000000e+00


attention softmax

In [82]:
# ============================================================
# ATTENTION SOFTMAX EQUIVALENCE
# ============================================================

attention_weights_cached = torch.softmax(
    scores_cached,
    dim=-1,
)

attention_weights_uncached = torch.softmax(
    scores_uncached,
    dim=-1,
)

diff = (
    attention_weights_cached.float()
    - attention_weights_uncached.float()
).abs()

print("=" * 70)
print("ATTENTION WEIGHTS EQUIVALENCE")
print("=" * 70)

print("cached:  ", attention_weights_cached.shape)
print("uncached:", attention_weights_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

print()
print("POSITION-BY-POSITION")

for pos in range(6):

    d = (
        attention_weights_cached[:, :, 0, pos].float()
        - attention_weights_uncached[:, :, 0, pos].float()
    ).abs()

    print(
        f"key position {pos}: "
        f"max={d.max().item():.8e} "
        f"mean={d.mean().item():.8e}"
    )

ATTENTION WEIGHTS EQUIVALENCE
cached:   torch.Size([1, 14, 1, 6])
uncached: torch.Size([1, 14, 1, 6])

MAX DIFF : 0.0
MEAN DIFF: 0.0

POSITION-BY-POSITION
key position 0: max=0.00000000e+00 mean=0.00000000e+00
key position 1: max=0.00000000e+00 mean=0.00000000e+00
key position 2: max=0.00000000e+00 mean=0.00000000e+00
key position 3: max=0.00000000e+00 mean=0.00000000e+00
key position 4: max=0.00000000e+00 mean=0.00000000e+00
key position 5: max=0.00000000e+00 mean=0.00000000e+00


A@V

In [84]:
# ============================================================
# CAPTURE UNCACHED V FROM NORMAL FORWARD
# ============================================================

layer_idx = 0
layer = adapter.model.model.layers[layer_idx]

uncached_v = {}

def capture_v(module, inputs, output):
    uncached_v["value"] = output.detach().clone()

hook = layer.self_attn.v_proj.register_forward_hook(capture_v)

# Full uncached sequence: prompt + generated token
with torch.no_grad():
    _ = adapter.model(
        input_ids=full_input_ids,
        use_cache=False,
    )

hook.remove()


# ============================================================
# RESHAPE UNCACHED V
# ============================================================

v_raw = uncached_v["value"]

B, T, _ = v_raw.shape

num_kv_heads = adapter.model.config.num_key_value_heads
head_dim = (
    adapter.model.config.hidden_size
    // adapter.model.config.num_attention_heads
)

v_uncached = (
    v_raw
    .view(B, T, num_kv_heads, head_dim)
    .transpose(1, 2)
    .contiguous()
)


# ============================================================
# CACHED V
# ============================================================

v_cached = (
    hf_cache.kv_cache.value_cache[layer_idx]
    [:, :, :T, :]
)


# ============================================================
# COMPARE
# ============================================================

print("=" * 70)
print("V NUMERICAL EQUIVALENCE — LAYER 0")
print("=" * 70)

print("cached:  ", v_cached.shape)
print("uncached:", v_uncached.shape)

diff = (
    v_cached.float()
    - v_uncached.float()
).abs()

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# POSITION-BY-POSITION
# ============================================================

print()
print("POSITION-BY-POSITION")
print("=" * 70)

for pos in range(T):

    diff = (
        v_cached[:, :, pos, :].float()
        - v_uncached[:, :, pos, :].float()
    ).abs()

    print(
        f"position {pos}: "
        f"max={diff.max().item():.8e} "
        f"mean={diff.mean().item():.8e}"
    )

V NUMERICAL EQUIVALENCE — LAYER 0
cached:   torch.Size([1, 2, 6, 64])
uncached: torch.Size([1, 2, 6, 64])

MAX DIFF : 0.0
MEAN DIFF: 0.0

POSITION-BY-POSITION
position 0: max=0.00000000e+00 mean=0.00000000e+00
position 1: max=0.00000000e+00 mean=0.00000000e+00
position 2: max=0.00000000e+00 mean=0.00000000e+00
position 3: max=0.00000000e+00 mean=0.00000000e+00
position 4: max=0.00000000e+00 mean=0.00000000e+00
position 5: max=0.00000000e+00 mean=0.00000000e+00


In [86]:
# ============================================================
# CAPTURE UNCACHED V — LAYER 0
# ============================================================

layer_idx = 0

uncached_v = {}

def capture_v(module, inputs, output):
    """
    Capture the V projection before attention.
    """
    v = output
    uncached_v["value"] = (
        v.detach()
        .clone()
        .view(
            1,
            -1,
            6,
            model.config.hidden_size
            // model.config.num_attention_heads,
        )
        .transpose(1, 2)
    )

# IMPORTANT:
# This must hook the actual v_proj of layer 0.
hook = (
    model.model.layers[layer_idx]
    .self_attn.v_proj
    .register_forward_hook(capture_v)
)

with torch.no_grad():
    _ = model(
        input_ids=full_input_ids,
        use_cache=False,
    )

hook.remove()

print("Captured uncached V:")
print(uncached_v["value"].shape)

Captured uncached V:
torch.Size([1, 6, 2, 64])


In [87]:
# ============================================================
# CAPTURE UNCACHED V — LAYER 0
# ============================================================

layer_idx = 0

uncached_v = {}

def capture_v(module, inputs, output):
    B, T, _ = output.shape

    v = output.view(
        B,
        T,
        model.config.num_key_value_heads,
        model.config.hidden_size
        // model.config.num_attention_heads,
    ).transpose(1, 2)

    uncached_v["value"] = v.detach().clone()


hook = (
    model.model.layers[layer_idx]
    .self_attn.v_proj
    .register_forward_hook(capture_v)
)

with torch.no_grad():
    _ = model(
        input_ids=full_input_ids,
        use_cache=False,
    )

hook.remove()

print("uncached V:", uncached_v["value"].shape)

uncached V: torch.Size([1, 2, 6, 64])


In [88]:
# ============================================================
# V NUMERICAL EQUIVALENCE — LAYER 0
# ============================================================

layer_idx = 0

# ------------------------------------------------------------
# Cached V
# ------------------------------------------------------------
v_cached = (
    hf_cache.kv_cache.value_cache[layer_idx]
    [:, :, :6, :]
)

# ------------------------------------------------------------
# Uncached V
# ------------------------------------------------------------
v_uncached = uncached_v["value"][:, :, :6, :]

print("=" * 70)
print("V NUMERICAL EQUIVALENCE — LAYER 0")
print("=" * 70)

print("cached V:  ", v_cached.shape)
print("uncached V:", v_uncached.shape)

diff = (
    v_cached.float()
    - v_uncached.float()
).abs()

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# POSITION-BY-POSITION
# ============================================================

print()
print("POSITION-BY-POSITION")
print("=" * 70)

for pos in range(6):

    diff = (
        v_cached[:, :, pos, :].float()
        - v_uncached[:, :, pos, :].float()
    ).abs()

    print(
        f"position {pos}: "
        f"max={diff.max().item():.8e} "
        f"mean={diff.mean().item():.8e}"
    )

V NUMERICAL EQUIVALENCE — LAYER 0
cached V:   torch.Size([1, 2, 6, 64])
uncached V: torch.Size([1, 2, 6, 64])

MAX DIFF : 0.0
MEAN DIFF: 0.0

POSITION-BY-POSITION
position 0: max=0.00000000e+00 mean=0.00000000e+00
position 1: max=0.00000000e+00 mean=0.00000000e+00
position 2: max=0.00000000e+00 mean=0.00000000e+00
position 3: max=0.00000000e+00 mean=0.00000000e+00
position 4: max=0.00000000e+00 mean=0.00000000e+00
position 5: max=0.00000000e+00 mean=0.00000000e+00


In [90]:
# ============================================================
# V NUMERICAL EQUIVALENCE — LAYER 0
# ============================================================

layer_idx = 0

v_cached = (
    hf_cache.kv_cache.value_cache[layer_idx]
    [:, :, :6, :]
)

v_uncached = uncached_v["value"][:, :, :6, :]

print("=" * 70)
print("V NUMERICAL EQUIVALENCE — LAYER 0")
print("=" * 70)

print("cached V:  ", v_cached.shape)
print("uncached V:", v_uncached.shape)

diff = (
    v_cached.float()
    - v_uncached.float()
).abs()

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

print()
print("POSITION-BY-POSITION")
print("=" * 70)

for pos in range(6):

    diff = (
        v_cached[:, :, pos, :].float()
        - v_uncached[:, :, pos, :].float()
    ).abs()

    print(
        f"position {pos}: "
        f"max={diff.max().item():.8e} "
        f"mean={diff.mean().item():.8e}"
    )

V NUMERICAL EQUIVALENCE — LAYER 0
cached V:   torch.Size([1, 2, 6, 64])
uncached V: torch.Size([1, 2, 6, 64])

MAX DIFF : 0.0
MEAN DIFF: 0.0

POSITION-BY-POSITION
position 0: max=0.00000000e+00 mean=0.00000000e+00
position 1: max=0.00000000e+00 mean=0.00000000e+00
position 2: max=0.00000000e+00 mean=0.00000000e+00
position 3: max=0.00000000e+00 mean=0.00000000e+00
position 4: max=0.00000000e+00 mean=0.00000000e+00
position 5: max=0.00000000e+00 mean=0.00000000e+00


In [91]:
# ============================================================
# ATTENTION CONTEXT EQUIVALENCE
# ============================================================

# ------------------------------------------------------------
# Expand V for GQA
# ------------------------------------------------------------

v_cached_gqa = v_cached.repeat_interleave(
    attention.num_key_value_groups,
    dim=1,
)

v_uncached_gqa = v_uncached.repeat_interleave(
    attention.num_key_value_groups,
    dim=1,
)

print("=" * 70)
print("V AFTER GQA")
print("=" * 70)

print("cached:  ", v_cached_gqa.shape)
print("uncached:", v_uncached_gqa.shape)


# ------------------------------------------------------------
# A @ V
# ------------------------------------------------------------

context_cached = torch.matmul(
    attention_weights_cached,
    v_cached_gqa,
)

context_uncached = torch.matmul(
    attention_weights_uncached,
    v_uncached_gqa,
)


# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------

diff = (
    context_cached.float()
    - context_uncached.float()
).abs()

print()
print("=" * 70)
print("ATTENTION CONTEXT EQUIVALENCE — A @ V")
print("=" * 70)

print("cached:  ", context_cached.shape)
print("uncached:", context_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

V AFTER GQA
cached:   torch.Size([1, 14, 6, 64])
uncached: torch.Size([1, 14, 6, 64])

ATTENTION CONTEXT EQUIVALENCE — A @ V
cached:   torch.Size([1, 14, 1, 64])
uncached: torch.Size([1, 14, 1, 64])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [92]:
# ============================================================
# BEFORE O_PROJ
# ============================================================

pre_o_cached = (
    context_cached
    .transpose(1, 2)
    .contiguous()
)

pre_o_uncached = (
    context_uncached
    .transpose(1, 2)
    .contiguous()
)

print("=" * 70)
print("BEFORE O_PROJ")
print("=" * 70)

print("cached:  ", pre_o_cached.shape)
print("uncached:", pre_o_uncached.shape)

diff = (
    pre_o_cached.float()
    - pre_o_uncached.float()
).abs()

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

BEFORE O_PROJ
cached:   torch.Size([1, 1, 14, 64])
uncached: torch.Size([1, 1, 14, 64])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [93]:
# ============================================================
# AFTER RESHAPE, BEFORE O_PROJ
# ============================================================

B, T, H, D = pre_o_cached.shape

flat_cached = pre_o_cached.view(
    B,
    T,
    H * D,
)

flat_uncached = pre_o_uncached.view(
    B,
    T,
    H * D,
)

diff = (
    flat_cached.float()
    - flat_uncached.float()
).abs()

print("=" * 70)
print("FLATTENED ATTENTION OUTPUT")
print("=" * 70)

print("cached:  ", flat_cached.shape)
print("uncached:", flat_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

FLATTENED ATTENTION OUTPUT
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [94]:
o_proj = (
    model.model.layers[0]
    .self_attn.o_proj
)

In [95]:
# ============================================================
# O_PROJ EQUIVALENCE
# ============================================================

with torch.no_grad():

    o_cached = o_proj(
        flat_cached
    )

    o_uncached = o_proj(
        flat_uncached
    )

diff = (
    o_cached.float()
    - o_uncached.float()
).abs()

print("=" * 70)
print("O_PROJ EQUIVALENCE")
print("=" * 70)

print("cached:  ", o_cached.shape)
print("uncached:", o_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

O_PROJ EQUIVALENCE
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0


test the input of the mlp

In [102]:
# ============================================================
# SETUP
# ============================================================




import gc
import torch

# Delete model-related Python objects
for name in [
    "adapter",
    "model",
    "attention",
    "hf_cache",
    "kv_cache",
    "full_output",
    "uncached_k",
    "uncached_v",
]:
    if name in globals():
        del globals()[name]

# Run Python garbage collection
gc.collect()

# Release PyTorch's cached CUDA memory
torch.cuda.empty_cache()

# Ask CUDA to release unused IPC memory
torch.cuda.ipc_collect()

print(torch.cuda.memory_summary())

adapter = ModelAdapter(
    model_name=config["model"]["model_name"],
    device=config["device"],
)

model = adapter.model
model.eval()

prompt = "The capital of France is"

prompt_ids = adapter.tokenize(prompt)

input_ids = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=adapter.device,
)

T = input_ids.shape[1]

print("prompt length:", T)
print("prompt ids:", prompt_ids)
print("input ids:", input_ids)


# ============================================================
# 1. UNCACHED PROMPT
# ============================================================

with torch.no_grad():
    prompt_output = model(
        input_ids=input_ids,
        use_cache=False,
    )

next_token = prompt_output.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

print()
print("next token:", next_token.item())


# ============================================================
# 2. CREATE CACHE
# ============================================================

kv_cache = KVCache(
    num_layers=model.config.num_hidden_layers,
    num_kv_heads=model.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model.config.hidden_size
        // model.config.num_attention_heads
    ),
    dtype=model.dtype,
    device=adapter.device,
)

hf_cache = EngineKVCache(kv_cache)

print()
print("cache length before prefill:",
      hf_cache.get_seq_length())


# ============================================================
# 3. CACHED PREFILL
# ============================================================

with torch.no_grad():
    prefill_output = model(
        input_ids=input_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )

print("cache length after prefill:",
      hf_cache.get_seq_length())

assert hf_cache.get_seq_length() == T


# ============================================================
# 4. CAPTURE LAYER-0 INPUT TO ATTENTION
# ============================================================

layer0 = model.model.layers[0]

captured = {}


def layer0_hook(module, inputs, output):

    # inputs[0] = hidden_states entering decoder layer
    hidden_states = inputs[0]

    captured["layer_input"] = (
        hidden_states.detach().clone()
    )


hook = layer0.register_forward_hook(layer0_hook)


# ============================================================
# 5. CACHED DECODE
# ============================================================

decode_position = T

position_ids = torch.tensor(
    [[decode_position]],
    dtype=torch.long,
    device=adapter.device,
)

with torch.no_grad():

    cached_decode_output = model(
        input_ids=next_token,
        position_ids=position_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )


hook.remove()


cached_layer_input = captured["layer_input"]


print()
print("=" * 70)
print("CACHED DECODE")
print("=" * 70)

print("cached layer-0 input:",
      cached_layer_input.shape)

print("cache length after decode:",
      hf_cache.get_seq_length())

assert hf_cache.get_seq_length() == T + 1


# ============================================================
# 6. UNCACHED FULL SEQUENCE
#
# [prompt tokens] + [generated token]
# ============================================================

full_input_ids = torch.cat(
    [
        input_ids,
        next_token,
    ],
    dim=1,
)

print()
print("full sequence shape:",
      full_input_ids.shape)


# ============================================================
# 7. CAPTURE LAYER-0 INPUT FOR UNCACHED FULL SEQUENCE
# ============================================================

captured = {}


def layer0_hook_uncached(module, inputs, output):

    hidden_states = inputs[0]

    captured["layer_input"] = (
        hidden_states.detach().clone()
    )


hook = layer0.register_forward_hook(
    layer0_hook_uncached
)


with torch.no_grad():

    full_output = model(
        input_ids=full_input_ids,
        use_cache=False,
    )


hook.remove()


uncached_layer_input = captured["layer_input"]


# ============================================================
# 8. COMPARE LAYER-0 INPUT
#
# We only care about the LAST token.
# ============================================================

cached_last = cached_layer_input[:, -1:, :]

uncached_last = uncached_layer_input[:, -1:, :]

diff = (
    cached_last.float()
    - uncached_last.float()
).abs()


print()
print("=" * 70)
print("LAYER-0 INPUT EQUIVALENCE")
print("=" * 70)

print("cached:  ", cached_last.shape)
print("uncached:", uncached_last.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# 9. COMPARE LAYER-0 INPUT
# WITH PREVIOUS PROMPT POSITION
#
# This is useful because the cached decode should represent
# exactly the same hidden state as full-sequence position T.
# ============================================================

print()
print("EXPECTED POSITION:", T)

assert cached_last.shape == uncached_last.shape


# ============================================================
# 10. COMPARE FINAL LOGITS
# ============================================================

cached_logits = (
    cached_decode_output.logits[:, -1, :]
)

uncached_logits = (
    full_output.logits[:, -1, :]
)

logit_diff = (
    cached_logits.float()
    - uncached_logits.float()
).abs()


print()
print("=" * 70)
print("FINAL LOGITS EQUIVALENCE")
print("=" * 70)

print("MAX DIFF :", logit_diff.max().item())
print("MEAN DIFF:", logit_diff.mean().item())

cached_token = cached_logits.argmax(
    dim=-1
)

uncached_token = uncached_logits.argmax(
    dim=-1
)

print()
print("cached token  :", cached_token.item())
print("uncached token:", uncached_token.item())


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    f"Layer-0 input max diff : "
    f"{diff.max().item():.8e}"
)

print(
    f"Layer-0 input mean diff: "
    f"{diff.mean().item():.8e}"
)

print(
    f"Logits max diff        : "
    f"{logit_diff.max().item():.8e}"
)

print(
    f"Logits mean diff       : "
    f"{logit_diff.mean().item():.8e}"
)

print(
    "Token match:",
    torch.equal(
        cached_token,
        uncached_token,
    )
)

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |    968 MiB |   2909 MiB |   9979 MiB |   9010 MiB |
|       from large pool |    956 MiB |   2837 MiB |   4741 MiB |   3785 MiB |
|       from small pool |     12 MiB |     73 MiB |   5238 MiB |   5225 MiB |
|---------------------------------------------------------------------------|
| Active memory         |    968 MiB |   2909 MiB |   9979 MiB |   9010 MiB |
|       from large pool |    956 MiB |   2837 MiB |   4741 MiB |

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

prompt length: 5
prompt ids: [785, 6722, 315, 9625, 374]
input ids: tensor([[ 785, 6722,  315, 9625,  374]], device='cuda:0')

next token: 12095

cache length before prefill: 0
cache length after prefill: 5

CACHED DECODE
cached layer-0 input: torch.Size([1, 1, 896])
cache length after decode: 6

full sequence shape: torch.Size([1, 6])

LAYER-0 INPUT EQUIVALENCE
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0

EXPECTED POSITION: 5

FINAL LOGITS EQUIVALENCE
MAX DIFF : 0.173583984375
MEAN DIFF: 0.027369718998670578

cached token  : 13
uncached token: 13

SUMMARY
Layer-0 input max diff : 0.00000000e+00
Layer-0 input mean diff: 0.00000000e+00
Logits max diff        : 1.73583984e-01
Logits mean diff       : 2.73697190e-02
Token match: True


In [105]:
# ============================================================
# FRESH CACHE — DO NOT REUSE THE OLD hf_cache
# ============================================================

kv_cache = KVCache(
    num_layers=model.config.num_hidden_layers,
    num_kv_heads=model.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model.config.hidden_size
        // model.config.num_attention_heads
    ),
    dtype=model.dtype,
    device=adapter.device,
)

hf_cache = EngineKVCache(kv_cache)

print("fresh cache length:", hf_cache.get_seq_length())


# ============================================================
# LAYER 0 MLP INPUT HOOK
# ============================================================

layer0 = model.model.layers[0]

cached_mlp_input = {}

def capture_mlp_input(storage):
    def hook(module, inputs, output):
        storage["value"] = inputs[0].detach().clone()
    return hook


hook = layer0.mlp.register_forward_hook(
    capture_mlp_input(cached_mlp_input)
)


# ============================================================
# CACHED PREFILL
# ============================================================

with torch.no_grad():
    _ = model(
        input_ids=input_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )

print("cache after prefill:", hf_cache.get_seq_length())


# ============================================================
# CACHED DECODE — EXACTLY ONCE
# ============================================================

with torch.no_grad():
    _ = model(
        input_ids=next_token,
        position_ids=torch.tensor(
            [[T]],
            dtype=torch.long,
            device=adapter.device,
        ),
        use_cache=True,
        past_key_values=hf_cache,
    )

print("cache after decode:", hf_cache.get_seq_length())

hook.remove()


# ============================================================
# GET CACHED MLP INPUT
# ============================================================

mlp_cached = cached_mlp_input["value"]

print()
print("=" * 70)
print("CACHED MLP INPUT")
print("=" * 70)
print("shape:", mlp_cached.shape)


# ============================================================
# UNCACHED FULL SEQUENCE
# ============================================================

full_sequence = torch.cat(
    [input_ids, next_token],
    dim=1,
)

uncached_mlp_input = {}

hook = layer0.mlp.register_forward_hook(
    capture_mlp_input(uncached_mlp_input)
)

with torch.no_grad():
    _ = model(
        input_ids=full_sequence,
        use_cache=False,
    )

hook.remove()


mlp_uncached = uncached_mlp_input["value"]


# ============================================================
# COMPARE ONLY THE FINAL TOKEN
# ============================================================

mlp_cached_last = mlp_cached[:, -1:, :]
mlp_uncached_last = mlp_uncached[:, -1:, :]

diff = (
    mlp_cached_last.float()
    - mlp_uncached_last.float()
).abs()


print()
print("=" * 70)
print("MLP INPUT EQUIVALENCE — LAYER 0")
print("=" * 70)

print("cached:  ", mlp_cached_last.shape)
print("uncached:", mlp_uncached_last.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

fresh cache length: 0
cache after prefill: 5
cache after decode: 6

CACHED MLP INPUT
shape: torch.Size([1, 1, 896])

MLP INPUT EQUIVALENCE — LAYER 0
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [108]:
# ============================================================
# FRESH CACHE
# ============================================================

kv_cache = KVCache(
    num_layers=model.config.num_hidden_layers,
    num_kv_heads=model.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model.config.hidden_size
        // model.config.num_attention_heads
    ),
    dtype=model.dtype,
    device=adapter.device,
)

hf_cache = EngineKVCache(kv_cache)


# ============================================================
# CAPTURE LAYER 0 MLP OUTPUT
# ============================================================

cached_mlp_output = {}

def capture_output(storage):
    def hook(module, inputs, output):
        storage["value"] = output.detach().clone()
    return hook


layer0 = model.model.layers[0]

hook = layer0.mlp.register_forward_hook(
    capture_output(cached_mlp_output)
)


# ============================================================
# CACHED PREFILL
# ============================================================

with torch.no_grad():
    _ = model(
        input_ids=input_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )


# ============================================================
# CACHED DECODE
# ============================================================

with torch.no_grad():
    _ = model(
        input_ids=next_token,
        position_ids=torch.tensor(
            [[T]],
            dtype=torch.long,
            device=adapter.device,
        ),
        use_cache=True,
        past_key_values=hf_cache,
    )

hook.remove()


# ============================================================
# UNCACHED FULL SEQUENCE
# ============================================================

full_sequence = torch.cat(
    [input_ids, next_token],
    dim=1,
)

uncached_mlp_output = {}

hook = layer0.mlp.register_forward_hook(
    capture_output(uncached_mlp_output)
)

with torch.no_grad():
    _ = model(
        input_ids=full_sequence,
        use_cache=False,
    )

hook.remove()


# ============================================================
# FINAL TOKEN ONLY
# ============================================================

mlp_cached = cached_mlp_output["value"][:, -1:, :]
mlp_uncached = uncached_mlp_output["value"][:, -1:, :]

diff = (
    mlp_cached.float()
    - mlp_uncached.float()
).abs()


print("=" * 70)
print("LAYER 0 MLP OUTPUT EQUIVALENCE")
print("=" * 70)

print("cached:  ", mlp_cached.shape)
print("uncached:", mlp_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

LAYER 0 MLP OUTPUT EQUIVALENCE
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275


#### gate projection test

In [111]:
# ============================================================
# LAYER 0 MLP — GATE PROJECTION EQUIVALENCE
# ============================================================

layer0 = model.model.layers[0]
mlp0 = layer0.mlp

cached_gate = {}
uncached_gate = {}


def capture_output(storage):
    def hook(module, inputs, output):
        storage["value"] = output.detach().clone()
    return hook


# ============================================================
# FRESH CACHE
# ============================================================

kv_cache = KVCache(
    num_layers=model.config.num_hidden_layers,
    num_kv_heads=model.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model.config.hidden_size
        // model.config.num_attention_heads
    ),
    dtype=model.dtype,
    device=adapter.device,
)

hf_cache = EngineKVCache(kv_cache)


# ============================================================
# CACHED
# ============================================================

hook = mlp0.gate_proj.register_forward_hook(
    capture_output(cached_gate)
)

with torch.no_grad():
    _ = model(
        input_ids=input_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )

with torch.no_grad():
    _ = model(
        input_ids=next_token,
        position_ids=torch.tensor(
            [[T]],
            dtype=torch.long,
            device=adapter.device,
        ),
        use_cache=True,
        past_key_values=hf_cache,
    )

hook.remove()


# ============================================================
# UNCACHED
# ============================================================

full_sequence = torch.cat(
    [input_ids, next_token],
    dim=1,
)

hook = mlp0.gate_proj.register_forward_hook(
    capture_output(uncached_gate)
)

with torch.no_grad():
    _ = model(
        input_ids=full_sequence,
        use_cache=False,
    )

hook.remove()


# ============================================================
# FINAL TOKEN
# ============================================================

gate_cached = cached_gate["value"][:, -1:, :]
gate_uncached = uncached_gate["value"][:, -1:, :]

diff = (
    gate_cached.float()
    - gate_uncached.float()
).abs()


print("=" * 70)
print("LAYER 0 GATE PROJECTION EQUIVALENCE")
print("=" * 70)

print("cached:  ", gate_cached.shape)
print("uncached:", gate_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

LAYER 0 GATE PROJECTION EQUIVALENCE
cached:   torch.Size([1, 1, 4864])
uncached: torch.Size([1, 1, 4864])

MAX DIFF : 4.76837158203125e-07
MEAN DIFF: 9.803395595309183e-11


In [114]:
# ============================================================
# LAYER 0 UP PROJECTION EQUIVALENCE
# FINAL TOKEN ONLY
# ============================================================

layer0 = model.model.layers[0]
mlp0 = layer0.mlp

# ------------------------------------------------------------
# Cached MLP input
# [1, 1, 896]
# ------------------------------------------------------------

mlp_input_cached = cached_mlp_input["value"]

# ------------------------------------------------------------
# Uncached MLP input
# [1, 6, 896]
#
# We only want the final token: position 5
# ------------------------------------------------------------

mlp_input_uncached = (
    uncached_mlp_input["value"][:, -1:, :]
)

print("=" * 70)
print("MLP INPUTS — FINAL TOKEN")
print("=" * 70)

print("cached:  ", mlp_input_cached.shape)
print("uncached:", mlp_input_uncached.shape)


# ------------------------------------------------------------
# Verify inputs first
# ------------------------------------------------------------

input_diff = (
    mlp_input_cached.float()
    - mlp_input_uncached.float()
).abs()

print()
print("INPUT MAX DIFF :", input_diff.max().item())
print("INPUT MEAN DIFF:", input_diff.mean().item())


# ------------------------------------------------------------
# UP PROJECTION
# ------------------------------------------------------------

with torch.no_grad():

    up_cached = mlp0.up_proj(
        mlp_input_cached
    )

    up_uncached = mlp0.up_proj(
        mlp_input_uncached
    )


# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------

diff = (
    up_cached.float()
    - up_uncached.float()
).abs()

print()
print("=" * 70)
print("LAYER 0 UP PROJECTION EQUIVALENCE")
print("=" * 70)

print("cached:  ", up_cached.shape)
print("uncached:", up_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

MLP INPUTS — FINAL TOKEN
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

INPUT MAX DIFF : 0.0
INPUT MEAN DIFF: 0.0

LAYER 0 UP PROJECTION EQUIVALENCE
cached:   torch.Size([1, 1, 4864])
uncached: torch.Size([1, 1, 4864])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [115]:
# ============================================================
# LAYER 0 — SwiGLU INTERMEDIATE EQUIVALENCE
# ============================================================

layer0 = model.model.layers[0]
mlp0 = layer0.mlp

# ------------------------------------------------------------
# Final-token MLP inputs
# ------------------------------------------------------------

x_cached = cached_mlp_input["value"]

x_uncached = (
    uncached_mlp_input["value"][:, -1:, :]
)

# ------------------------------------------------------------
# Gate + Up projections
# ------------------------------------------------------------

with torch.no_grad():

    gate_cached = mlp0.gate_proj(x_cached)
    gate_uncached = mlp0.gate_proj(x_uncached)

    up_cached = mlp0.up_proj(x_cached)
    up_uncached = mlp0.up_proj(x_uncached)


# ------------------------------------------------------------
# SiLU
# ------------------------------------------------------------

silu_cached = torch.nn.functional.silu(
    gate_cached
)

silu_uncached = torch.nn.functional.silu(
    gate_uncached
)


# ------------------------------------------------------------
# SwiGLU intermediate
# ------------------------------------------------------------

intermediate_cached = (
    silu_cached * up_cached
)

intermediate_uncached = (
    silu_uncached * up_uncached
)


# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------

diff = (
    intermediate_cached.float()
    - intermediate_uncached.float()
).abs()


print("=" * 70)
print("LAYER 0 SwiGLU INTERMEDIATE EQUIVALENCE")
print("=" * 70)

print("cached:  ", intermediate_cached.shape)
print("uncached:", intermediate_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

LAYER 0 SwiGLU INTERMEDIATE EQUIVALENCE
cached:   torch.Size([1, 1, 4864])
uncached: torch.Size([1, 1, 4864])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [116]:
# ============================================================
# LAYER 0 — DOWN PROJECTION EQUIVALENCE
# ============================================================

with torch.no_grad():

    down_cached = mlp0.down_proj(
        intermediate_cached
    )

    down_uncached = mlp0.down_proj(
        intermediate_uncached
    )

diff = (
    down_cached.float()
    - down_uncached.float()
).abs()

print("=" * 70)
print("LAYER 0 DOWN PROJECTION EQUIVALENCE")
print("=" * 70)

print("cached:  ", down_cached.shape)
print("uncached:", down_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

LAYER 0 DOWN PROJECTION EQUIVALENCE
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [117]:
# ============================================================
# LAYER 0 OUTPUT EQUIVALENCE
# ============================================================

layer_idx = 0
layer0 = model.model.layers[layer_idx]

cached_layer0_output = {}
uncached_layer0_output = {}


def capture_layer_output(storage):
    def hook(module, inputs, output):
        # Qwen2DecoderLayer returns:
        #   (hidden_states, ...)
        if isinstance(output, tuple):
            storage["value"] = output[0].detach().clone()
        else:
            storage["value"] = output.detach().clone()

    return hook


# ============================================================
# FRESH CACHE
# ============================================================

kv_cache = KVCache(
    num_layers=model.config.num_hidden_layers,
    num_kv_heads=model.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model.config.hidden_size
        // model.config.num_attention_heads
    ),
    dtype=model.dtype,
    device=adapter.device,
)

hf_cache = EngineKVCache(kv_cache)


# ============================================================
# CACHED — PREFILL
# ============================================================

with torch.no_grad():
    _ = model(
        input_ids=input_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )


# ============================================================
# CACHED — DECODE
# ============================================================

hook = layer0.register_forward_hook(
    capture_layer_output(cached_layer0_output)
)

with torch.no_grad():
    _ = model(
        input_ids=next_token,
        position_ids=torch.tensor(
            [[T]],
            dtype=torch.long,
            device=adapter.device,
        ),
        use_cache=True,
        past_key_values=hf_cache,
    )

hook.remove()


# ============================================================
# UNCACHED — FULL SEQUENCE
# ============================================================

full_sequence = torch.cat(
    [input_ids, next_token],
    dim=1,
)

hook = layer0.register_forward_hook(
    capture_layer_output(uncached_layer0_output)
)

with torch.no_grad():
    _ = model(
        input_ids=full_sequence,
        use_cache=False,
    )

hook.remove()


# ============================================================
# FINAL TOKEN
# ============================================================

layer0_cached = cached_layer0_output["value"][:, -1:, :]
layer0_uncached = uncached_layer0_output["value"][:, -1:, :]


diff = (
    layer0_cached.float()
    - layer0_uncached.float()
).abs()


# ============================================================
# RESULT
# ============================================================

print("=" * 70)
print("LAYER 0 OUTPUT EQUIVALENCE")
print("=" * 70)

print("cached:  ", layer0_cached.shape)
print("uncached:", layer0_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

LAYER 0 OUTPUT EQUIVALENCE
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.00390625
MEAN DIFF: 0.00021962609025649726


In [118]:
# ============================================================
# LAYER 0 — PRE-MLP RESIDUAL EQUIVALENCE
#
# This is the tensor entering post_attention_layernorm:
#
# h_attn_res = residual + attention_output
#
# If this differs while MLP input is identical, then LayerNorm
# is hiding the difference.
# ============================================================

layer_idx = 0
layer0 = model.model.layers[layer_idx]

cached_residual = {}
uncached_residual = {}


def capture_input(storage):
    def hook(module, inputs):
        storage["value"] = inputs[0].detach().clone()
    return hook


# ============================================================
# FRESH CACHE
# ============================================================

kv_cache = KVCache(
    num_layers=model.config.num_hidden_layers,
    num_kv_heads=model.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model.config.hidden_size
        // model.config.num_attention_heads
    ),
    dtype=model.dtype,
    device=adapter.device,
)

hf_cache = EngineKVCache(kv_cache)


# ============================================================
# CACHED
# ============================================================

hook = layer0.post_attention_layernorm.register_forward_pre_hook(
    capture_input(cached_residual)
)

with torch.no_grad():

    # Prefill
    _ = model(
        input_ids=input_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )

    # Decode
    _ = model(
        input_ids=next_token,
        position_ids=torch.tensor(
            [[T]],
            dtype=torch.long,
            device=adapter.device,
        ),
        use_cache=True,
        past_key_values=hf_cache,
    )

hook.remove()


# ============================================================
# UNCACHED
# ============================================================

full_sequence = torch.cat(
    [input_ids, next_token],
    dim=1,
)

hook = layer0.post_attention_layernorm.register_forward_pre_hook(
    capture_input(uncached_residual)
)

with torch.no_grad():
    _ = model(
        input_ids=full_sequence,
        use_cache=False,
    )

hook.remove()


# ============================================================
# FINAL TOKEN
# ============================================================

res_cached = cached_residual["value"][:, -1:, :]
res_uncached = uncached_residual["value"][:, -1:, :]


diff = (
    res_cached.float()
    - res_uncached.float()
).abs()


print("=" * 70)
print("LAYER 0 PRE-MLP RESIDUAL EQUIVALENCE")
print("=" * 70)

print("cached:  ", res_cached.shape)
print("uncached:", res_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# POSITION-WISE FEATURE CHECK
# ============================================================

print()
print("LARGEST DIFFERING FEATURES")
print("=" * 70)

flat_diff = diff.flatten()

top_values, top_indices = torch.topk(
    flat_diff,
    k=min(10, flat_diff.numel()),
)

for value, index in zip(top_values, top_indices):
    print(
        f"feature {index.item():4d}: "
        f"diff={value.item():.8e}"
    )

LAYER 0 PRE-MLP RESIDUAL EQUIVALENCE
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0

LARGEST DIFFERING FEATURES
feature    7: diff=0.00000000e+00
feature    6: diff=0.00000000e+00
feature    4: diff=0.00000000e+00
feature    5: diff=0.00000000e+00
feature    1: diff=0.00000000e+00
feature    0: diff=0.00000000e+00
feature    2: diff=0.00000000e+00
feature    3: diff=0.00000000e+00
feature    8: diff=0.00000000e+00
feature    9: diff=0.00000000e+00


In [121]:
# ============================================================
# LAYER 0 — MANUAL MLP FINAL-TOKEN EQUIVALENCE
# ============================================================

layer_idx = 0
layer0 = model.model.layers[layer_idx]
mlp0 = layer0.mlp

# ------------------------------------------------------------
# Extract tensors from hook storage
# ------------------------------------------------------------

x_cached = cached_mlp_input["value"][:, -1:, :]
x_uncached = uncached_mlp_input["value"][:, -1:, :]

print("=" * 70)
print("FINAL-TOKEN MLP INPUTS")
print("=" * 70)

print("cached:  ", x_cached.shape)
print("uncached:", x_uncached.shape)

diff = (
    x_cached.float()
    - x_uncached.float()
).abs()

print("INPUT MAX DIFF :", diff.max().item())
print("INPUT MEAN DIFF:", diff.mean().item())


# ============================================================
# GATE PROJECTION
# ============================================================

gate_cached = layer0.mlp.gate_proj(x_cached)
gate_uncached = layer0.mlp.gate_proj(x_uncached)

diff = (
    gate_cached.float()
    - gate_uncached.float()
).abs()

print()
print("=" * 70)
print("GATE PROJECTION")
print("=" * 70)

print("cached:  ", gate_cached.shape)
print("uncached:", gate_uncached.shape)

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# UP PROJECTION
# ============================================================

up_cached = layer0.mlp.up_proj(x_cached)
up_uncached = layer0.mlp.up_proj(x_uncached)

diff = (
    up_cached.float()
    - up_uncached.float()
).abs()

print()
print("=" * 70)
print("UP PROJECTION")
print("=" * 70)

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# SWIGLU INTERMEDIATE
# ============================================================

intermediate_cached = (
    torch.nn.functional.silu(gate_cached)
    * up_cached
)

intermediate_uncached = (
    torch.nn.functional.silu(gate_uncached)
    * up_uncached
)

diff = (
    intermediate_cached.float()
    - intermediate_uncached.float()
).abs()

print()
print("=" * 70)
print("SWIGLU INTERMEDIATE")
print("=" * 70)

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# DOWN PROJECTION
# ============================================================

down_cached = layer0.mlp.down_proj(
    intermediate_cached
)

down_uncached = layer0.mlp.down_proj(
    intermediate_uncached
)

diff = (
    down_cached.float()
    - down_uncached.float()
).abs()

print()
print("=" * 70)
print("DOWN PROJECTION")
print("=" * 70)

print("cached:  ", down_cached.shape)
print("uncached:", down_uncached.shape)

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# ACTUAL MLP
# ============================================================

actual_cached = layer0.mlp(x_cached)
actual_uncached = layer0.mlp(x_uncached)

diff = (
    actual_cached.float()
    - actual_uncached.float()
).abs()

print()
print("=" * 70)
print("ACTUAL MLP OUTPUT")
print("=" * 70)

print("cached:  ", actual_cached.shape)
print("uncached:", actual_uncached.shape)

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

FINAL-TOKEN MLP INPUTS
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])
INPUT MAX DIFF : 0.0
INPUT MEAN DIFF: 0.0

GATE PROJECTION
cached:   torch.Size([1, 1, 4864])
uncached: torch.Size([1, 1, 4864])
MAX DIFF : 0.0
MEAN DIFF: 0.0

UP PROJECTION
MAX DIFF : 0.0
MEAN DIFF: 0.0

SWIGLU INTERMEDIATE
MAX DIFF : 0.0
MEAN DIFF: 0.0

DOWN PROJECTION
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])
MAX DIFF : 0.0
MEAN DIFF: 0.0

ACTUAL MLP OUTPUT
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])
MAX DIFF : 0.0
MEAN DIFF: 0.0


In [122]:
# ============================================================
# SETUP
# ============================================================




import gc
import torch

# Delete model-related Python objects
for name in [
    "adapter",
    "model",
    "attention",
    "hf_cache",
    "kv_cache",
    "full_output",
    "uncached_k",
    "uncached_v",
]:
    if name in globals():
        del globals()[name]

# Run Python garbage collection
gc.collect()

# Release PyTorch's cached CUDA memory
torch.cuda.empty_cache()

# Ask CUDA to release unused IPC memory
torch.cuda.ipc_collect()

print(torch.cuda.memory_summary())

adapter = ModelAdapter(
    model_name=config["model"]["model_name"],
    device=config["device"],
)

model = adapter.model
model.eval()

prompt = "The capital of France is"

prompt_ids = adapter.tokenize(prompt)

input_ids = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=adapter.device,
)

T = input_ids.shape[1]

print("prompt length:", T)
print("prompt ids:", prompt_ids)
print("input ids:", input_ids)


# ============================================================
# 1. UNCACHED PROMPT
# ============================================================

with torch.no_grad():
    prompt_output = model(
        input_ids=input_ids,
        use_cache=False,
    )

next_token = prompt_output.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

print()
print("next token:", next_token.item())


# ============================================================
# 2. CREATE CACHE
# ============================================================

kv_cache = KVCache(
    num_layers=model.config.num_hidden_layers,
    num_kv_heads=model.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model.config.hidden_size
        // model.config.num_attention_heads
    ),
    dtype=model.dtype,
    device=adapter.device,
)

hf_cache = EngineKVCache(kv_cache)

print()
print("cache length before prefill:",
      hf_cache.get_seq_length())


# ============================================================
# 3. CACHED PREFILL
# ============================================================

with torch.no_grad():
    prefill_output = model(
        input_ids=input_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )

print("cache length after prefill:",
      hf_cache.get_seq_length())

assert hf_cache.get_seq_length() == T


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  83741 KiB |   2909 MiB |  13715 MiB |  13633 MiB |
|       from large pool |  80023 KiB |   2837 MiB |   5717 MiB |   5639 MiB |
|       from small pool |   3718 KiB |     73 MiB |   7998 MiB |   7994 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  83741 KiB |   2909 MiB |  13715 MiB |  13633 MiB |
|       from large pool |  80023 KiB |   2837 MiB |   5717 MiB |

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

prompt length: 5
prompt ids: [785, 6722, 315, 9625, 374]
input ids: tensor([[ 785, 6722,  315, 9625,  374]], device='cuda:0')

next token: 12095

cache length before prefill: 0
cache length after prefill: 5


In [123]:
# ============================================================
# 4. CAPTURE LAYER-0 MLP STAGES
# ============================================================

layer0 = model.model.layers[0]
mlp0 = layer0.mlp

captures = {
    "mlp_input": {},
    "gate": {},
    "up": {},
    "mlp_output": {},
}


def capture_input(storage):
    def hook(module, inputs):
        storage["value"] = inputs[0].detach().clone()
    return hook


def capture_output(storage):
    def hook(module, inputs, output):
        storage["value"] = output.detach().clone()
    return hook


hooks = []

hooks.append(
    layer0.register_forward_pre_hook(
        capture_input(captures["mlp_input"])
    )
)

hooks.append(
    mlp0.gate_proj.register_forward_hook(
        capture_output(captures["gate"])
    )
)

hooks.append(
    mlp0.up_proj.register_forward_hook(
        capture_output(captures["up"])
    )
)

hooks.append(
    mlp0.register_forward_hook(
        capture_output(captures["mlp_output"])
    )
)


# ============================================================
# 5. CACHED DECODE
# ============================================================

with torch.no_grad():
    cached_output = model(
        input_ids=next_token,
        position_ids=torch.tensor(
            [[T]],
            dtype=torch.long,
            device=adapter.device,
        ),
        use_cache=True,
        past_key_values=hf_cache,
    )


for hook in hooks:
    hook.remove()


print("=" * 70)
print("CACHED DECODE")
print("=" * 70)

print("cache length:", hf_cache.get_seq_length())

for name, storage in captures.items():
    print(
        f"{name:15s}:",
        storage["value"].shape
    )

CACHED DECODE
cache length: 6
mlp_input      : torch.Size([1, 1, 896])
gate           : torch.Size([1, 1, 4864])
up             : torch.Size([1, 1, 4864])
mlp_output     : torch.Size([1, 1, 896])


In [124]:
# ============================================================
# 6. CACHED MLP INTERMEDIATE
# ============================================================

x_cached = captures["mlp_input"]["value"]

gate_cached = captures["gate"]["value"]
up_cached = captures["up"]["value"]

swiglu_cached = (
    torch.nn.functional.silu(gate_cached)
    * up_cached
)

down_cached = mlp0.down_proj(
    swiglu_cached
)

print("=" * 70)
print("CACHED MLP STAGES")
print("=" * 70)

print("input :", x_cached.shape)
print("gate  :", gate_cached.shape)
print("up    :", up_cached.shape)
print("swiglu:", swiglu_cached.shape)
print("down  :", down_cached.shape)
print("actual:", captures["mlp_output"]["value"].shape)

CACHED MLP STAGES
input : torch.Size([1, 1, 896])
gate  : torch.Size([1, 1, 4864])
up    : torch.Size([1, 1, 4864])
swiglu: torch.Size([1, 1, 4864])
down  : torch.Size([1, 1, 896])
actual: torch.Size([1, 1, 896])


In [125]:
actual_cached = captures["mlp_output"]["value"]

diff = (
    down_cached.float()
    - actual_cached.float()
).abs()

print()
print("CACHED MANUAL vs ACTUAL MLP")
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


CACHED MANUAL vs ACTUAL MLP
MAX DIFF : 0.0
MEAN DIFF: 0.0


In [126]:
# ============================================================
# 7. UNCACHED FULL SEQUENCE
# ============================================================

full_sequence = torch.cat(
    [input_ids, next_token],
    dim=1,
)

uncached_captures = {
    "mlp_input": {},
    "gate": {},
    "up": {},
    "mlp_output": {},
}


hooks = []

hooks.append(
    layer0.register_forward_pre_hook(
        capture_input(uncached_captures["mlp_input"])
    )
)

hooks.append(
    mlp0.gate_proj.register_forward_hook(
        capture_output(uncached_captures["gate"])
    )
)

hooks.append(
    mlp0.up_proj.register_forward_hook(
        capture_output(uncached_captures["up"])
    )
)

hooks.append(
    mlp0.register_forward_hook(
        capture_output(uncached_captures["mlp_output"])
    )
)


with torch.no_grad():
    uncached_output = model(
        input_ids=full_sequence,
        use_cache=False,
    )


for hook in hooks:
    hook.remove()


print("=" * 70)
print("UNCACHED FULL SEQUENCE")
print("=" * 70)

for name, storage in uncached_captures.items():
    print(
        f"{name:15s}:",
        storage["value"].shape
    )

UNCACHED FULL SEQUENCE
mlp_input      : torch.Size([1, 6, 896])
gate           : torch.Size([1, 6, 4864])
up             : torch.Size([1, 6, 4864])
mlp_output     : torch.Size([1, 6, 896])


In [127]:
# ============================================================
# 8. FINAL TOKEN ONLY
# ============================================================

x_uncached = (
    uncached_captures["mlp_input"]["value"]
    [:, -1:, :]
)

gate_uncached = (
    uncached_captures["gate"]["value"]
    [:, -1:, :]
)

up_uncached = (
    uncached_captures["up"]["value"]
    [:, -1:, :]
)

actual_uncached = (
    uncached_captures["mlp_output"]["value"]
    [:, -1:, :]
)


print("=" * 70)
print("FINAL TOKEN MLP TENSORS")
print("=" * 70)

print("cached input  :", x_cached.shape)
print("uncached input:", x_uncached.shape)

print("cached gate   :", gate_cached.shape)
print("uncached gate :", gate_uncached.shape)

FINAL TOKEN MLP TENSORS
cached input  : torch.Size([1, 1, 896])
uncached input: torch.Size([1, 1, 896])
cached gate   : torch.Size([1, 1, 4864])
uncached gate : torch.Size([1, 1, 4864])


In [128]:
# ============================================================
# 9. MLP INPUT
# ============================================================

diff = (
    x_cached.float()
    - x_uncached.float()
).abs()

print("=" * 70)
print("MLP INPUT EQUIVALENCE")
print("=" * 70)

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

MLP INPUT EQUIVALENCE
MAX DIFF : 0.0
MEAN DIFF: 0.0


In [129]:
# ============================================================
# 10. GATE PROJECTION
# ============================================================

diff = (
    gate_cached.float()
    - gate_uncached.float()
).abs()

print("=" * 70)
print("GATE PROJECTION EQUIVALENCE")
print("=" * 70)

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

GATE PROJECTION EQUIVALENCE
MAX DIFF : 4.76837158203125e-07
MEAN DIFF: 9.803395595309183e-11


In [130]:
# ============================================================
# 11. UP PROJECTION
# ============================================================

up_cached = mlp0.up_proj(x_cached)
up_uncached = mlp0.up_proj(x_uncached)

diff = (
    up_cached.float()
    - up_uncached.float()
).abs()

print("=" * 70)
print("UP PROJECTION EQUIVALENCE")
print("=" * 70)

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

UP PROJECTION EQUIVALENCE
MAX DIFF : 0.0
MEAN DIFF: 0.0


In [131]:
# ============================================================
# 12. SWIGLU INTERMEDIATE
# ============================================================

swiglu_cached = (
    torch.nn.functional.silu(gate_cached)
    * up_cached
)

swiglu_uncached = (
    torch.nn.functional.silu(gate_uncached)
    * up_uncached
)

diff = (
    swiglu_cached.float()
    - swiglu_uncached.float()
).abs()

print("=" * 70)
print("SWIGLU INTERMEDIATE EQUIVALENCE")
print("=" * 70)

print("cached:  ", swiglu_cached.shape)
print("uncached:", swiglu_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

SWIGLU INTERMEDIATE EQUIVALENCE
cached:   torch.Size([1, 1, 4864])
uncached: torch.Size([1, 1, 4864])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [132]:
# ============================================================
# 13. DOWN PROJECTION
# ============================================================

down_cached = mlp0.down_proj(
    swiglu_cached
)

down_uncached = mlp0.down_proj(
    swiglu_uncached
)

diff = (
    down_cached.float()
    - down_uncached.float()
).abs()

print("=" * 70)
print("DOWN PROJECTION EQUIVALENCE")
print("=" * 70)

print("cached:  ", down_cached.shape)
print("uncached:", down_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

DOWN PROJECTION EQUIVALENCE
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0


In [133]:
# ============================================================
# 14. ACTUAL MLP OUTPUT
# ============================================================

diff = (
    captures["mlp_output"]["value"].float()
    - actual_uncached.float()
).abs()

print("=" * 70)
print("ACTUAL MLP OUTPUT EQUIVALENCE")
print("=" * 70)

print("cached:  ", captures["mlp_output"]["value"].shape)
print("uncached:", actual_uncached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

ACTUAL MLP OUTPUT EQUIVALENCE
cached:   torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275


In [134]:
# ============================================================
# 15. MANUAL vs ACTUAL
# ============================================================

manual_cached = down_cached
manual_uncached = down_uncached

actual_cached = captures["mlp_output"]["value"]
actual_uncached = (
    uncached_captures["mlp_output"]["value"]
    [:, -1:, :]
)


cached_internal_diff = (
    manual_cached.float()
    - actual_cached.float()
).abs()

uncached_internal_diff = (
    manual_uncached.float()
    - actual_uncached.float()
).abs()


print("=" * 70)
print("CACHED: MANUAL vs ACTUAL")
print("=" * 70)

print(
    "MAX DIFF :",
    cached_internal_diff.max().item()
)

print(
    "MEAN DIFF:",
    cached_internal_diff.mean().item()
)


print()
print("=" * 70)
print("UNCACHED: MANUAL vs ACTUAL")
print("=" * 70)

print(
    "MAX DIFF :",
    uncached_internal_diff.max().item()
)

print(
    "MEAN DIFF:",
    uncached_internal_diff.mean().item()
)

CACHED: MANUAL vs ACTUAL
MAX DIFF : 2.23773193359375
MEAN DIFF: 0.17486028373241425

UNCACHED: MANUAL vs ACTUAL
MAX DIFF : 2.23773193359375
MEAN DIFF: 0.1748591959476471


In [135]:
# ============================================================
# ISOLATE QWEN MLP IMPLEMENTATION — LAYER 0
# ============================================================

layer_idx = 0
layer0 = model.model.layers[layer_idx]
mlp0 = layer0.mlp

# ------------------------------------------------------------
# Get FINAL TOKEN inputs
# ------------------------------------------------------------

x_cached = cached_mlp_input["value"][:, -1:, :]
x_uncached = uncached_mlp_input["value"][:, -1:, :]

print("=" * 70)
print("MLP INPUT")
print("=" * 70)

print("cached:", x_cached.shape)
print("uncached:", x_uncached.shape)

input_diff = (
    x_cached.float() - x_uncached.float()
).abs()

print("MAX DIFF :", input_diff.max().item())
print("MEAN DIFF:", input_diff.mean().item())


# ============================================================
# DIRECT MODEL MLP
# ============================================================

with torch.no_grad():

    actual_cached = mlp0(x_cached)
    actual_uncached = mlp0(x_uncached)


# ============================================================
# MANUAL MLP USING THE MODEL'S OWN COMPONENTS
# ============================================================

with torch.no_grad():

    gate = mlp0.gate_proj(x_cached)
    up = mlp0.up_proj(x_cached)

    # IMPORTANT:
    # use the EXACT activation function from the model
    activated_gate = mlp0.act_fn(gate)

    intermediate = activated_gate * up

    manual_output = mlp0.down_proj(intermediate)


# ============================================================
# COMPARE MANUAL VS DIRECT
# ============================================================

diff = (
    manual_output.float()
    - actual_cached.float()
).abs()

print()
print("=" * 70)
print("MANUAL MLP vs DIRECT mlp0(x)")
print("=" * 70)

print("manual :", manual_output.shape)
print("direct :", actual_cached.shape)

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# CHECK EACH INTERNAL STEP
# ============================================================

# Direct components

with torch.no_grad():

    direct_gate = mlp0.gate_proj(x_cached)
    direct_up = mlp0.up_proj(x_cached)

    direct_intermediate = (
        mlp0.act_fn(direct_gate) * direct_up
    )

    direct_down = mlp0.down_proj(direct_intermediate)


print()
print("=" * 70)
print("INTERNAL COMPONENT CHECK")
print("=" * 70)


diff_gate = (
    gate.float() - direct_gate.float()
).abs()

print("GATE")
print("MAX :", diff_gate.max().item())
print("MEAN:", diff_gate.mean().item())


diff_up = (
    up.float() - direct_up.float()
).abs()

print()
print("UP")
print("MAX :", diff_up.max().item())
print("MEAN:", diff_up.mean().item())


diff_intermediate = (
    intermediate.float()
    - direct_intermediate.float()
).abs()

print()
print("SWIGLU")
print("MAX :", diff_intermediate.max().item())
print("MEAN:", diff_intermediate.mean().item())


diff_down = (
    manual_output.float()
    - direct_down.float()
).abs()

print()
print("DOWN")
print("MAX :", diff_down.max().item())
print("MEAN:", diff_down.mean().item())


# ============================================================
# MOST IMPORTANT:
# CHECK WHAT mlp0.forward ACTUALLY DOES
# ============================================================

print()
print("=" * 70)
print("MLP ACTIVATION")
print("=" * 70)

print("act_fn:", mlp0.act_fn)
print("gate_proj:", mlp0.gate_proj)
print("up_proj:", mlp0.up_proj)
print("down_proj:", mlp0.down_proj)

MLP INPUT
cached: torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])
MAX DIFF : 0.0
MEAN DIFF: 0.0

MANUAL MLP vs DIRECT mlp0(x)
manual : torch.Size([1, 1, 896])
direct : torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0

INTERNAL COMPONENT CHECK
GATE
MAX : 0.0
MEAN: 0.0

UP
MAX : 0.0
MEAN: 0.0

SWIGLU
MAX : 0.0
MEAN: 0.0

DOWN
MAX : 0.0
MEAN: 0.0

MLP ACTIVATION
act_fn: SiLUActivation()
gate_proj: Linear(in_features=896, out_features=4864, bias=False)
up_proj: Linear(in_features=896, out_features=4864, bias=False)
down_proj: Linear(in_features=4864, out_features=896, bias=False)


In [136]:
# ============================================================
# DIRECT MLP vs CAPTURED MLP OUTPUT
# ============================================================

layer_idx = 0
layer0 = model.model.layers[layer_idx]
mlp0 = layer0.mlp

# ------------------------------------------------------------
# Extract final-token MLP inputs
# ------------------------------------------------------------

x_cached = cached_mlp_input["value"][:, -1:, :]
x_uncached = uncached_mlp_input["value"][:, -1:, :]

# ------------------------------------------------------------
# Direct MLP computation
# ------------------------------------------------------------

with torch.no_grad():
    direct_cached = mlp0(x_cached)

with torch.no_grad():
    direct_uncached = mlp0(x_uncached)

# ------------------------------------------------------------
# Captured MLP outputs
# ------------------------------------------------------------

captured_cached = cached_mlp_output["value"][:, -1:, :]
captured_uncached = uncached_mlp_output["value"][:, -1:, :]


# ============================================================
# CACHED: DIRECT vs CAPTURED
# ============================================================

diff_cached = (
    direct_cached.float()
    - captured_cached.float()
).abs()

print("=" * 70)
print("CACHED — DIRECT MLP vs CAPTURED MLP")
print("=" * 70)

print("direct  :", direct_cached.shape)
print("captured:", captured_cached.shape)

print()
print("MAX DIFF :", diff_cached.max().item())
print("MEAN DIFF:", diff_cached.mean().item())


# ============================================================
# UNCACHED: DIRECT vs CAPTURED
# ============================================================

diff_uncached = (
    direct_uncached.float()
    - captured_uncached.float()
).abs()

print()
print("=" * 70)
print("UNCACHED — DIRECT MLP vs CAPTURED MLP")
print("=" * 70)

print("direct  :", direct_uncached.shape)
print("captured:", captured_uncached.shape)

print()
print("MAX DIFF :", diff_uncached.max().item())
print("MEAN DIFF:", diff_uncached.mean().item())


# ============================================================
# DIRECT CACHED vs DIRECT UNCACHED
# ============================================================

diff_direct = (
    direct_cached.float()
    - direct_uncached.float()
).abs()

print()
print("=" * 70)
print("DIRECT CACHED MLP vs DIRECT UNCACHED MLP")
print("=" * 70)

print("MAX DIFF :", diff_direct.max().item())
print("MEAN DIFF:", diff_direct.mean().item())


# ============================================================
# CAPTURED CACHED vs CAPTURED UNCACHED
# ============================================================

diff_captured = (
    captured_cached.float()
    - captured_uncached.float()
).abs()

print()
print("=" * 70)
print("CAPTURED CACHED MLP vs CAPTURED UNCACHED MLP")
print("=" * 70)

print("MAX DIFF :", diff_captured.max().item())
print("MEAN DIFF:", diff_captured.mean().item())

CACHED — DIRECT MLP vs CAPTURED MLP
direct  : torch.Size([1, 1, 896])
captured: torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0

UNCACHED — DIRECT MLP vs CAPTURED MLP
direct  : torch.Size([1, 1, 896])
captured: torch.Size([1, 1, 896])

MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275

DIRECT CACHED MLP vs DIRECT UNCACHED MLP
MAX DIFF : 0.0
MEAN DIFF: 0.0

CAPTURED CACHED MLP vs CAPTURED UNCACHED MLP
MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275


In [137]:
# ============================================================
# MLP REPEATABILITY TEST — UNCACHED FINAL TOKEN
# ============================================================

x = uncached_mlp_input["value"][:, -1:, :].detach().clone()

print("=" * 70)
print("MLP REPEATABILITY — UNCACHED FINAL TOKEN")
print("=" * 70)

print("input:", x.shape)
print("dtype:", x.dtype)


with torch.no_grad():

    y1 = mlp0(x).detach().clone()
    y2 = mlp0(x).detach().clone()
    y3 = mlp0(x).detach().clone()


# ============================================================
# COMPARE REPEATED DIRECT CALLS
# ============================================================

def report_diff(name, a, b):

    diff = (
        a.float() - b.float()
    ).abs()

    print()
    print(name)
    print("MAX DIFF :", diff.max().item())
    print("MEAN DIFF:", diff.mean().item())


report_diff(
    "DIRECT CALL 1 vs DIRECT CALL 2",
    y1,
    y2,
)

report_diff(
    "DIRECT CALL 2 vs DIRECT CALL 3",
    y2,
    y3,
)

report_diff(
    "DIRECT CALL 1 vs CAPTURED",
    y1,
    uncached_mlp_output["value"][:, -1:, :],
)

MLP REPEATABILITY — UNCACHED FINAL TOKEN
input: torch.Size([1, 1, 896])
dtype: torch.bfloat16

DIRECT CALL 1 vs DIRECT CALL 2
MAX DIFF : 0.0
MEAN DIFF: 0.0

DIRECT CALL 2 vs DIRECT CALL 3
MAX DIFF : 0.0
MEAN DIFF: 0.0

DIRECT CALL 1 vs CAPTURED
MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275


In [138]:
# ============================================================
# FRESH UNCACHED MLP CONSISTENCY TEST
# ============================================================

layer_idx = 0
layer0 = model.model.layers[layer_idx]
mlp0 = layer0.mlp

full_sequence = torch.cat(
    [input_ids, next_token],
    dim=1,
)

fresh = {
    "input": None,
    "output": None,
}


# ------------------------------------------------------------
# Capture MLP input
# ------------------------------------------------------------

def capture_mlp_input(module, inputs):
    fresh["input"] = inputs[0].detach().clone()


# ------------------------------------------------------------
# Capture MLP output
# ------------------------------------------------------------

def capture_mlp_output(module, inputs, output):
    fresh["output"] = output.detach().clone()


input_hook = mlp0.register_forward_pre_hook(
    capture_mlp_input
)

output_hook = mlp0.register_forward_hook(
    capture_mlp_output
)


# ------------------------------------------------------------
# ONE FRESH UNCACHED FORWARD
# ------------------------------------------------------------

with torch.no_grad():

    fresh_output = model(
        input_ids=full_sequence,
        use_cache=False,
    )


# ------------------------------------------------------------
# Remove hooks
# ------------------------------------------------------------

input_hook.remove()
output_hook.remove()


# ============================================================
# EXTRACT FINAL TOKEN
# ============================================================

x = fresh["input"][:, -1:, :]
y_captured = fresh["output"][:, -1:, :]


print("=" * 70)
print("FRESH UNCACHED MLP CAPTURE")
print("=" * 70)

print("MLP input :", x.shape)
print("MLP output:", y_captured.shape)

print("input dtype :", x.dtype)
print("output dtype:", y_captured.dtype)


# ============================================================
# DIRECT MLP ON THE EXACT CAPTURED INPUT
# ============================================================

with torch.no_grad():

    y_direct = mlp0(x)


# ============================================================
# DIRECT vs CAPTURED
# ============================================================

diff = (
    y_direct.float()
    - y_captured.float()
).abs()

print()
print("=" * 70)
print("DIRECT MLP vs CAPTURED MLP")
print("=" * 70)

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# CHECK INPUT AGAINST ITSELF
# ============================================================

x_clone = x.detach().clone()

input_diff = (
    x.float()
    - x_clone.float()
).abs()

print()
print("=" * 70)
print("INPUT SANITY")
print("=" * 70)

print("MAX DIFF :", input_diff.max().item())
print("MEAN DIFF:", input_diff.mean().item())


# ============================================================
# MANUAL MLP
# ============================================================

with torch.no_grad():

    gate = mlp0.gate_proj(x)
    up = mlp0.up_proj(x)

    intermediate = (
        mlp0.act_fn(gate) * up
    )

    y_manual = mlp0.down_proj(intermediate)


diff_manual = (
    y_manual.float()
    - y_direct.float()
).abs()

print()
print("=" * 70)
print("MANUAL MLP vs DIRECT MLP")
print("=" * 70)

print("MAX DIFF :", diff_manual.max().item())
print("MEAN DIFF:", diff_manual.mean().item())


diff_manual_capture = (
    y_manual.float()
    - y_captured.float()
).abs()

print()
print("=" * 70)
print("MANUAL MLP vs CAPTURED MLP")
print("=" * 70)

print("MAX DIFF :", diff_manual_capture.max().item())
print("MEAN DIFF:", diff_manual_capture.mean().item())

FRESH UNCACHED MLP CAPTURE
MLP input : torch.Size([1, 1, 896])
MLP output: torch.Size([1, 1, 896])
input dtype : torch.bfloat16
output dtype: torch.bfloat16

DIRECT MLP vs CAPTURED MLP
MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275

INPUT SANITY
MAX DIFF : 0.0
MEAN DIFF: 0.0

MANUAL MLP vs DIRECT MLP
MAX DIFF : 0.0
MEAN DIFF: 0.0

MANUAL MLP vs CAPTURED MLP
MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275


In [139]:
# ============================================================
# INSPECT THE ACTUAL MLP IMPLEMENTATION
# ============================================================

import inspect

layer0 = model.model.layers[0]
mlp0 = layer0.mlp

print("=" * 70)
print("MLP IMPLEMENTATION")
print("=" * 70)

print("MLP class:")
print(type(mlp0))

print()
print("MLP forward:")
print(inspect.getsource(mlp0.forward))

print()
print("=" * 70)
print("MODULE STATE")
print("=" * 70)

print("training:", mlp0.training)
print("dtype:", next(mlp0.parameters()).dtype)
print("device:", next(mlp0.parameters()).device)

print()
print("=" * 70)
print("MODEL STATE")
print("=" * 70)

print("model.training:", model.training)

print()
print("=" * 70)
print("COMPILE / OPTIMIZATION")
print("=" * 70)

print("model type:", type(model))

print(
    "has _compiled_call_impl:",
    hasattr(mlp0, "_compiled_call_impl")
)

print(
    "compiled call:",
    getattr(mlp0, "_compiled_call_impl", None)
)

MLP IMPLEMENTATION
MLP class:
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2MLP'>

MLP forward:
    def forward(self, x):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        return down_proj


MODULE STATE
training: False
dtype: torch.bfloat16
device: cuda:0

MODEL STATE
model.training: False

COMPILE / OPTIMIZATION
model type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
has _compiled_call_impl: True
compiled call: None


In [140]:
# ============================================================
# MLP INPUT — STORAGE / LAYOUT EQUIVALENCE
# ============================================================

x_cached = cached_mlp_input["value"]
x_uncached = uncached_mlp_input["value"]

# Make sure we are comparing only the final token
x_cached = x_cached[:, -1:, :]
x_uncached = x_uncached[:, -1:, :]

print("=" * 70)
print("MLP INPUT STORAGE / LAYOUT")
print("=" * 70)

print()
print("SHAPE")
print("cached  :", x_cached.shape)
print("uncached:", x_uncached.shape)

print()
print("STRIDE")
print("cached  :", x_cached.stride())
print("uncached:", x_uncached.stride())

print()
print("CONTIGUOUS")
print("cached  :", x_cached.is_contiguous())
print("uncached:", x_uncached.is_contiguous())

print()
print("STORAGE OFFSET")
print("cached  :", x_cached.storage_offset())
print("uncached:", x_uncached.storage_offset())

print()
print("DATA POINTER")
print("cached  :", x_cached.data_ptr())
print("uncached:", x_uncached.data_ptr())

print()
print("DTYPE")
print("cached  :", x_cached.dtype)
print("uncached:", x_uncached.dtype)

print()
print("DEVICE")
print("cached  :", x_cached.device)
print("uncached:", x_uncached.device)

print()
print("VALUE DIFFERENCE")

diff = (
    x_cached.float()
    - x_uncached.float()
).abs()

print("MAX :", diff.max().item())
print("MEAN:", diff.mean().item())

MLP INPUT STORAGE / LAYOUT

SHAPE
cached  : torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

STRIDE
cached  : (896, 896, 1)
uncached: (5376, 896, 1)

CONTIGUOUS
cached  : True
uncached: True

STORAGE OFFSET
cached  : 0
uncached: 4480

DATA POINTER
cached  : 140039240046592
uncached: 140035793736960

DTYPE
cached  : torch.bfloat16
uncached: torch.bfloat16

DEVICE
cached  : cuda:0
uncached: cuda:0

VALUE DIFFERENCE
MAX : 0.0
MEAN: 0.0


In [141]:
# ============================================================
# MLP LAYOUT TEST
# ============================================================

mlp0 = model.model.layers[0].mlp

# ------------------------------------------------------------
# Original uncached final-token tensor
# ------------------------------------------------------------

x_uncached = uncached_mlp_input["value"][:, -1:, :]

# ------------------------------------------------------------
# Create a fresh tensor with the same physical layout
# as the cached MLP input
# ------------------------------------------------------------

x_uncached_contiguous = x_uncached.contiguous()

# ------------------------------------------------------------
# Cached tensor
# ------------------------------------------------------------

x_cached = cached_mlp_input["value"][:, -1:, :]


print("=" * 70)
print("MLP INPUT LAYOUT TEST")
print("=" * 70)

print()
print("ORIGINAL UNCACHED")
print("shape :", x_uncached.shape)
print("stride:", x_uncached.stride())
print("offset:", x_uncached.storage_offset())

print()
print("CONTIGUOUS UNCACHED")
print("shape :", x_uncached_contiguous.shape)
print("stride:", x_uncached_contiguous.stride())
print("offset:", x_uncached_contiguous.storage_offset())

print()
print("CACHED")
print("shape :", x_cached.shape)
print("stride:", x_cached.stride())
print("offset:", x_cached.storage_offset())


# ============================================================
# VALUE EQUIVALENCE
# ============================================================

print()
print("=" * 70)
print("INPUT VALUE EQUIVALENCE")
print("=" * 70)

diff = (
    x_cached.float()
    - x_uncached_contiguous.float()
).abs()

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# MLP
# ============================================================

with torch.no_grad():

    y_cached = mlp0(x_cached)

    y_uncached_original = mlp0(
        x_uncached
    )

    y_uncached_contiguous = mlp0(
        x_uncached_contiguous
    )


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 70)
print("CACHED vs ORIGINAL UNCACHED")
print("=" * 70)

diff = (
    y_cached.float()
    - y_uncached_original.float()
).abs()

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


print()
print("=" * 70)
print("CACHED vs CONTIGUOUS UNCACHED")
print("=" * 70)

diff = (
    y_cached.float()
    - y_uncached_contiguous.float()
).abs()

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


print()
print("=" * 70)
print("ORIGINAL UNCACHED vs CONTIGUOUS UNCACHED")
print("=" * 70)

diff = (
    y_uncached_original.float()
    - y_uncached_contiguous.float()
).abs()

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

MLP INPUT LAYOUT TEST

ORIGINAL UNCACHED
shape : torch.Size([1, 1, 896])
stride: (5376, 896, 1)
offset: 4480

CONTIGUOUS UNCACHED
shape : torch.Size([1, 1, 896])
stride: (5376, 896, 1)
offset: 4480

CACHED
shape : torch.Size([1, 1, 896])
stride: (896, 896, 1)
offset: 0

INPUT VALUE EQUIVALENCE
MAX DIFF : 0.0
MEAN DIFF: 0.0

CACHED vs ORIGINAL UNCACHED
MAX DIFF : 0.0
MEAN DIFF: 0.0

CACHED vs CONTIGUOUS UNCACHED
MAX DIFF : 0.0
MEAN DIFF: 0.0

ORIGINAL UNCACHED vs CONTIGUOUS UNCACHED
MAX DIFF : 0.0
MEAN DIFF: 0.0


In [142]:
# ============================================================
# SELF-CONTAINED LAYER-0 MLP FORWARD EQUIVALENCE
# ============================================================

layer_idx = 0
layer0 = model.model.layers[layer_idx]
mlp0 = layer0.mlp

cached_capture = {}
uncached_capture = {}


def make_hook(storage):
    def hook(module, inputs, output):

        storage["input"] = inputs[0].detach().clone()
        storage["output"] = output.detach().clone()

    return hook


# ============================================================
# FRESH CACHE
# ============================================================

kv_cache = KVCache(
    num_layers=model.config.num_hidden_layers,
    num_kv_heads=model.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model.config.hidden_size
        // model.config.num_attention_heads
    ),
    dtype=model.dtype,
    device=adapter.device,
)

hf_cache = EngineKVCache(kv_cache)


# ============================================================
# CACHED FORWARD
# ============================================================

hook = mlp0.register_forward_hook(
    make_hook(cached_capture)
)

with torch.no_grad():

    # Prefill
    _ = model(
        input_ids=input_ids,
        use_cache=True,
        past_key_values=hf_cache,
    )

    # Decode
    _ = model(
        input_ids=next_token,
        position_ids=torch.tensor(
            [[T]],
            dtype=torch.long,
            device=adapter.device,
        ),
        use_cache=True,
        past_key_values=hf_cache,
    )

hook.remove()


# ============================================================
# UNCACHED FORWARD
# ============================================================

full_sequence = torch.cat(
    [input_ids, next_token],
    dim=1,
)

hook = mlp0.register_forward_hook(
    make_hook(uncached_capture)
)

with torch.no_grad():

    _ = model(
        input_ids=full_sequence,
        use_cache=False,
    )

hook.remove()


# ============================================================
# SELECT FINAL TOKEN
# ============================================================

cached_input = cached_capture["input"][:, -1:, :]
cached_output = cached_capture["output"][:, -1:, :]

uncached_input = uncached_capture["input"][:, -1:, :]
uncached_output = uncached_capture["output"][:, -1:, :]


# ============================================================
# INPUT
# ============================================================

print("=" * 70)
print("MLP INPUT — SAME FORWARD CAPTURE")
print("=" * 70)

print("cached  :", cached_input.shape)
print("uncached:", uncached_input.shape)

diff = (
    cached_input.float()
    - uncached_input.float()
).abs()

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# OUTPUT
# ============================================================

print()
print("=" * 70)
print("MLP OUTPUT — SAME FORWARD CAPTURE")
print("=" * 70)

print("cached  :", cached_output.shape)
print("uncached:", uncached_output.shape)

diff = (
    cached_output.float()
    - uncached_output.float()
).abs()

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


# ============================================================
# DIRECT RECOMPUTATION
# ============================================================

with torch.no_grad():

    direct_cached = mlp0(cached_input)

    direct_uncached = mlp0(uncached_input)


print()
print("=" * 70)
print("CACHED — CAPTURED vs DIRECT")
print("=" * 70)

diff = (
    cached_output.float()
    - direct_cached.float()
).abs()

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


print()
print("=" * 70)
print("UNCACHED — CAPTURED vs DIRECT")
print("=" * 70)

diff = (
    uncached_output.float()
    - direct_uncached.float()
).abs()

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())


print()
print("=" * 70)
print("DIRECT CACHED vs DIRECT UNCACHED")
print("=" * 70)

diff = (
    direct_cached.float()
    - direct_uncached.float()
).abs()

print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

MLP INPUT — SAME FORWARD CAPTURE
cached  : torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.0
MEAN DIFF: 0.0

MLP OUTPUT — SAME FORWARD CAPTURE
cached  : torch.Size([1, 1, 896])
uncached: torch.Size([1, 1, 896])

MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275

CACHED — CAPTURED vs DIRECT
MAX DIFF : 0.0
MEAN DIFF: 0.0

UNCACHED — CAPTURED vs DIRECT
MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275

DIRECT CACHED vs DIRECT UNCACHED
MAX DIFF : 0.0
MEAN DIFF: 0.0


In [143]:
# ============================================================
# FULL-SHAPE vs FINAL-TOKEN MLP
# ============================================================

layer0 = model.model.layers[0]
mlp0 = layer0.mlp

# ------------------------------------------------------------
# Use the exact uncached MLP input captured from the full
# sequence forward.
# ------------------------------------------------------------

x_full = uncached_capture["input"]

x_last = x_full[:, -1:, :]

print("=" * 70)
print("INPUTS")
print("=" * 70)

print("full :", x_full.shape)
print("last :", x_last.shape)

print()
print("FINAL TOKEN INPUT SANITY")

diff = (
    x_full[:, -1:, :].float()
    - x_last.float()
).abs()

print("MAX :", diff.max().item())
print("MEAN:", diff.mean().item())


# ============================================================
# MLP ON FULL SEQUENCE
# ============================================================

with torch.no_grad():
    y_full = mlp0(x_full)


# ============================================================
# MLP ON FINAL TOKEN ONLY
# ============================================================

with torch.no_grad():
    y_last = mlp0(x_last)


# ============================================================
# COMPARE FINAL TOKEN
# ============================================================

y_full_last = y_full[:, -1:, :]

print()
print("=" * 70)
print("FULL-SEQUENCE MLP vs FINAL-TOKEN MLP")
print("=" * 70)

print("full final :", y_full_last.shape)
print("last only  :", y_last.shape)

diff = (
    y_full_last.float()
    - y_last.float()
).abs()

print()
print("MAX DIFF :", diff.max().item())
print("MEAN DIFF:", diff.mean().item())

INPUTS
full : torch.Size([1, 6, 896])
last : torch.Size([1, 1, 896])

FINAL TOKEN INPUT SANITY
MAX : 0.0
MEAN: 0.0

FULL-SEQUENCE MLP vs FINAL-TOKEN MLP
full final : torch.Size([1, 1, 896])
last only  : torch.Size([1, 1, 896])

MAX DIFF : 0.00390625
MEAN DIFF: 0.00021693324379157275


In [144]:
# ============================================================
# FP32 CACHED vs UNCACHED NUMERICAL EQUIVALENCE
# ============================================================

import gc
import torch

# ============================================================
# 0. CLEANUP
# ============================================================

for name in [
    "adapter_fp32",
    "model_fp32",
    "kv_cache_fp32",
    "hf_cache_fp32",
    "prompt_output_fp32",
    "prefill_output_fp32",
    "cached_output_fp32",
    "uncached_output_fp32",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("=" * 70)
print("GPU MEMORY BEFORE FP32 MODEL")
print("=" * 70)

print(
    f"allocated : "
    f"{torch.cuda.memory_allocated() / 1024**2:.2f} MB"
)

print(
    f"reserved  : "
    f"{torch.cuda.memory_reserved() / 1024**2:.2f} MB"
)


# ============================================================
# 1. LOAD MODEL
# ============================================================

try:

    adapter_fp32 = ModelAdapter(
        model_name=config["model"]["model_name"],
        device=config["device"],
    )

    model_fp32 = adapter_fp32.model

    # --------------------------------------------------------
    # Explicit FP32
    # --------------------------------------------------------

    model_fp32 = model_fp32.float()
    model_fp32.eval()

    print()
    print("=" * 70)
    print("MODEL")
    print("=" * 70)

    print("dtype:", next(model_fp32.parameters()).dtype)
    print("device:", next(model_fp32.parameters()).device)

except torch.cuda.OutOfMemoryError:

    print()
    print("=" * 70)
    print("FP32 MODEL OOM")
    print("=" * 70)

    print(
        "The RTX 3050 does not have enough VRAM to load "
        "the model in FP32."
    )

    raise


# ============================================================
# 2. PROMPT
# ============================================================

prompt = "The capital of France is"

prompt_ids = adapter_fp32.tokenize(prompt)

input_ids_fp32 = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=adapter_fp32.device,
)

T = input_ids_fp32.shape[1]

print()
print("=" * 70)
print("PROMPT")
print("=" * 70)

print("prompt length:", T)
print("prompt ids:", prompt_ids)
print("input ids:", input_ids_fp32)


# ============================================================
# 3. UNCACHED PROMPT
# ============================================================

with torch.no_grad():

    prompt_output_fp32 = model_fp32(
        input_ids=input_ids_fp32,
        use_cache=False,
    )

next_token_fp32 = prompt_output_fp32.logits[:, -1, :].argmax(
    dim=-1,
    keepdim=True,
)

print()
print("=" * 70)
print("NEXT TOKEN")
print("=" * 70)

print("next token:", next_token_fp32.item())


# ============================================================
# 4. CREATE FP32 KV CACHE
# ============================================================

kv_cache_fp32 = KVCache(
    num_layers=model_fp32.config.num_hidden_layers,
    num_kv_heads=model_fp32.config.num_key_value_heads,
    max_seq_len=T + 1,
    head_dim=(
        model_fp32.config.hidden_size
        // model_fp32.config.num_attention_heads
    ),
    dtype=torch.float32,
    device=adapter_fp32.device,
)

hf_cache_fp32 = EngineKVCache(kv_cache_fp32)

print()
print("=" * 70)
print("CACHE")
print("=" * 70)

print(
    "cache length before prefill:",
    hf_cache_fp32.get_seq_length(),
)


# ============================================================
# 5. CACHED PREFILL
# ============================================================

try:

    with torch.no_grad():

        prefill_output_fp32 = model_fp32(
            input_ids=input_ids_fp32,
            use_cache=True,
            past_key_values=hf_cache_fp32,
        )

except torch.cuda.OutOfMemoryError:

    print()
    print("=" * 70)
    print("FP32 PREFILL OOM")
    print("=" * 70)

    raise


print(
    "cache length after prefill:",
    hf_cache_fp32.get_seq_length(),
)

assert hf_cache_fp32.get_seq_length() == T


# ============================================================
# 6. CACHED DECODE
# ============================================================

with torch.no_grad():

    cached_output_fp32 = model_fp32(
        input_ids=next_token_fp32,
        position_ids=torch.tensor(
            [[T]],
            dtype=torch.long,
            device=adapter_fp32.device,
        ),
        use_cache=True,
        past_key_values=hf_cache_fp32,
    )

print(
    "cache length after decode:",
    hf_cache_fp32.get_seq_length(),
)

assert hf_cache_fp32.get_seq_length() == T + 1


# ============================================================
# 7. UNCACHED FULL SEQUENCE
# ============================================================

full_sequence_fp32 = torch.cat(
    [
        input_ids_fp32,
        next_token_fp32,
    ],
    dim=1,
)

with torch.no_grad():

    uncached_output_fp32 = model_fp32(
        input_ids=full_sequence_fp32,
        use_cache=False,
    )


# ============================================================
# 8. FINAL TOKEN LOGITS
# ============================================================

logits_cached_fp32 = cached_output_fp32.logits[:, -1, :]

logits_uncached_fp32 = uncached_output_fp32.logits[:, -1, :]


# ============================================================
# 9. NUMERICAL EQUIVALENCE
# ============================================================

diff = (
    logits_cached_fp32
    - logits_uncached_fp32
).abs()


print()
print("=" * 70)
print("FP32 FINAL LOGITS EQUIVALENCE")
print("=" * 70)

print("cached logits  :", logits_cached_fp32.shape)
print("uncached logits:", logits_uncached_fp32.shape)

print()
print(
    "MAX DIFF :",
    diff.max().item(),
)

print(
    "MEAN DIFF:",
    diff.mean().item(),
)


# ============================================================
# 10. TOKEN EQUIVALENCE
# ============================================================

cached_token_fp32 = logits_cached_fp32.argmax(
    dim=-1
).item()

uncached_token_fp32 = logits_uncached_fp32.argmax(
    dim=-1
).item()


print()
print("=" * 70)
print("FP32 TOKEN EQUIVALENCE")
print("=" * 70)

print("cached token  :", cached_token_fp32)
print("uncached token:", uncached_token_fp32)

print(
    "TOKEN MATCH:",
    cached_token_fp32 == uncached_token_fp32,
)


# ============================================================
# 11. LOGIT STATISTICS
# ============================================================

print()
print("=" * 70)
print("FP32 LOGIT STATISTICS")
print("=" * 70)

print(
    "cached max:",
    logits_cached_fp32.max().item(),
)

print(
    "cached min:",
    logits_cached_fp32.min().item(),
)

print(
    "uncached max:",
    logits_uncached_fp32.max().item(),
)

print(
    "uncached min:",
    logits_uncached_fp32.min().item(),
)


# ============================================================
# 12. GPU MEMORY
# ============================================================

print()
print("=" * 70)
print("GPU MEMORY AFTER TEST")
print("=" * 70)

print(
    f"allocated : "
    f"{torch.cuda.memory_allocated() / 1024**2:.2f} MB"
)

print(
    f"reserved  : "
    f"{torch.cuda.memory_reserved() / 1024**2:.2f} MB"
)

GPU MEMORY BEFORE FP32 MODEL
allocated : 1020.42 MB
reserved  : 1086.00 MB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


MODEL
dtype: torch.float32
device: cuda:0

PROMPT
prompt length: 5
prompt ids: [785, 6722, 315, 9625, 374]
input ids: tensor([[ 785, 6722,  315, 9625,  374]], device='cuda:0')

NEXT TOKEN
next token: 12095

CACHE
cache length before prefill: 0
cache length after prefill: 5
cache length after decode: 6

FP32 FINAL LOGITS EQUIVALENCE
cached logits  : torch.Size([1, 151936])
uncached logits: torch.Size([1, 151936])

MAX DIFF : 1.239776611328125e-05
MEAN DIFF: 1.8670865529202274e-06

FP32 TOKEN EQUIVALENCE
cached token  : 13
uncached token: 13
TOKEN MATCH: True

FP32 LOGIT STATISTICS
cached max: 20.463897705078125
cached min: -11.957040786743164
uncached max: 20.463897705078125
uncached min: -11.95704460144043

GPU MEMORY AFTER TEST
allocated : 2929.87 MB
reserved  : 3014.00 MB
